### Choose where this notebook runs

This is the **IQM edition**: the 2- and 3-qubit Grover searches, the sudoku and triangle-finding circuits
(and the runs of the In-depth analysis) are transpiled to the IQM native gate set **PRX + CZ** and run on
**Garnet** (20 qubits) or **Emerald** (54 qubits) via IQM Resonance. Pick the backend by editing the single
`TARGET` line below (see `README.md` and `iqm_backend.py` in this folder):

| `TARGET` | runs on | token? | QPU credits? |
|---|---|---|---|
| `"fake_apollo"` | local 20-qubit noisy simulator | no | no |
| `"garnet:mock"` / `"emerald:mock"` | IQM Resonance cloud mock (pipeline check: returns RANDOM bitstrings, no physics) | yes | no |
| `"garnet"` / `"emerald"` | **real QPU** (20q / 54q) | yes | yes |


# Grover's Algorithm — the full textbook chapter

**Course**: Quantum Computing Laboratory
**Version**: 2.2 (Qiskit 2.x)
**Source**: the *Grover's Algorithm* chapter of the Qiskit Textbook, ported to the Qiskit 2.x API

---

## 🎯 Where this notebook sits

This is the second of the three Grover notebooks of the study path:

| notebook | what it does |
|---|---|
| `14_Simple_Grover` | the minimal version: 2- and 3-qubit searches with a hand-built oracle and diffusion |
| **`15_Grover_Algorithm`** (this one) | the complete textbook chapter: the geometry of amplitude amplification, the 2-qubit example checked on the state vector, a 3-qubit example with two marked states, runs on real hardware, and Grover as a *problem solver* — a 2×2 sudoku turned into a satisfiability oracle (clauses, phase kickback, uncomputation) and the triangle-finding problem on a W-state search space |
| `16_Grover_with_Primitives` | the production route: the library `grover_operator`, the Sampler primitive and the Qiskit-patterns workflow on IBM hardware |

Every code cell is followed by a **🔧 Reading the code** table, the maths behind the numbers is worked out in **📐 Working it out** boxes, and the **🔬 In-depth analysis** at the end solves two of the author's problems (a general Grover solver, the number of iterations) and measures what the sudoku costs on hardware.


In [ ]:
from iqm_backend import get_backend, run_counts   # helper module in this folder (see iqm_backend.py)
# run_counts(circuit, TARGET, shots=...) = transpile to the IQM native gates (PRX + CZ) + run + get_counts()
# get_backend(TARGET) returns the raw Qiskit backend object if you want to call backend.run yourself
TARGET = "fake_apollo"   # change this one line to choose the backend (see the table above)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `from iqm_backend import get_backend, run_counts` | the helper module of this folder: `run_counts(circuit, TARGET, shots=...)` = `transpile_to_IQM` (rewrite the circuit with the native `r` = PRX rotations and `cz` gates, routed on the chip's square lattice) → `backend.run` → counts as a plain dict; `run_counts(..., return_transpiled=True)` also returns the transpiled circuit |
| `TARGET = "fake_apollo"` | a local noisy simulator with the 20-qubit Garnet topology: no token, no credits, realistic gate and readout noise; `"garnet:mock"` checks the cloud pipeline but returns RANDOM bitstrings; `"garnet"` / `"emerald"` are the real QPUs (token + credits; IQM Resonance declines circuits above 10 000 native instructions) |
| `get_backend(TARGET)` | the raw Qiskit backend object (name, qubit count, coupling map), used in Sections 2.1.2 and 3.1.2 |

</details>


In this section, we introduce Grover's algorithm and how it can be used to solve unstructured search problems. We then implement Grover's algorithm on actual problems using Qiskit, run on a simulator and an actual device.


## Contents

1. [Introduction](#introduction)    
   1.1 [Algorithm overview](#overview)   
   1.2 [Grover Step by step](#steps)    
   1.3 [State preparation](#state-prep)    
   1.4 [The Oracle](#oracle)    
   1.5 [The diffusion operator](#diffusion)    
2. [Example: 2 Qubits](#2qubits)    
   2.1 [Simulation](#2qubits-simulation)    
   2.2 [Device](#2qubits-device)    
3. [Example: 3 Qubits](#3qubits)     
   3.1 [Simulation](#3qubits-simulation)    
   3.2 [Device](#3qubits-device)    
4. [Problems](#problems)    
   4.1 [Solving Sudoku using Grover's Algorithm](#sudoku)    
   4.2 [Solving the triangle problem using Grover's Algorithm](#tri)        
5. [References](#references)


## 1. Introduction <a id='introduction'></a>

You have likely heard that one of the many advantages a quantum computer has over a classical computer is its superior speed searching databases. Grover's algorithm demonstrates this capability. This algorithm can speed up an unstructured search problem quadratically, but its uses extend beyond that; it can serve as a general trick or subroutine to obtain quadratic run time improvements for a variety of other algorithms. This is called the amplitude amplification trick.    


Suppose you are given a large list of $N$ items. Among these items there is one item with a unique property that we wish to locate; we will call this one the winner $w$. Think of each item in the list as a box of a particular color. Say all items in the list are gray except the winner $w$, which is purple. 



![image1](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_list.png)

To find the purple box -- the *marked item* -- using classical computation, one would have to check on average $N/2$ of these boxes, and in the worst case, all $N$ of them. On a quantum computer, however, we can find the marked item in roughly $\sqrt{N}$ steps with Grover's amplitude amplification trick. A quadratic speedup is indeed a substantial time-saver for finding marked items in long lists. Additionally, the algorithm does not use the list's internal structure, which makes it *generic;* this is why it immediately provides a quadratic quantum speed-up for many classical problems.


### 1.1 Algorithm Overview <a id='overview'></a>


Grover's algorithm consists of three main algorithms steps: state preparation, the oracle, and the diffusion operator. The state preparation is where we create the search space, which is all possible cases the answer could take. In the list example we mentioned above, the search space would be all the items of that list.    
The oracle is what marks the correct answer, or answers we are looking for, and the diffusion operator magnifies these answers so they can stand out and be measured at the end of the algorithm.

![image2](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_steps.png)    


So how does the algorithm work? Before looking at the list of items, we have no idea where the marked item is. Therefore, any guess of its location is as good as any other, which can be expressed in terms of a
uniform superposition: $|s \rangle = \frac{1}{\sqrt{N}} \sum_{x = 0}^{N -1} | x
\rangle.$

If at this point we were to measure in the standard basis $ \{ | x \rangle \} $, this superposition would collapse, according to the fifth quantum law, to any one of the basis states with the same probability of $\frac{1}{N} = \frac{1}{2^n}$. Our chances of guessing the right value $w$ is therefore $1$ in $2^n$, as could be expected. Hence, on average we would need to try about $N/2 = 2^{n-1}$ times to guess the correct item.

Enter the procedure called amplitude amplification, which is how a quantum computer significantly enhances this probability. This procedure stretches out (amplifies) the amplitude of the marked item, which shrinks the other items' amplitude, so that measuring the final state will return the right item with near-certainty. 

This algorithm has a nice geometrical interpretation in terms of two reflections, which generate a rotation in a two-dimensional plane. The only two special states we need to consider are the winner $| w \rangle$ and the uniform superposition $| s \rangle$. These two vectors span a two-dimensional plane in the vector space $\mathbb{C}^N$. They are not quite perpendicular because $| w \rangle$ occurs in the superposition with amplitude $N^{-1/2}$ as well.
We can, however, introduce an additional state $|s'\rangle$ that is in the span of these two vectors, which is perpendicular to $| w \rangle$ and is obtained from $|s \rangle$ by removing $| w \rangle$ and
rescaling. 

**Step 1**: The amplitude amplification procedure starts out in the uniform superposition $| s \rangle$, which is easily constructed from $| s \rangle = H^{\otimes n} | 0 \rangle^n$ or using another symmetric entangled states.

![image3](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_step1.jpg)


The left graphic corresponds to the two-dimensional plane spanned by perpendicular vectors $|w\rangle$ and $|s'\rangle$ which allows to express the initial state as $|s\rangle = \sin \theta | w \rangle + \cos \theta | s' \rangle,$ where $\theta = \arcsin \langle s | w \rangle = \arcsin \frac{1}{\sqrt{N}}$. The right graphic is a bar graph of the amplitudes of the state $| s \rangle$.

**Step 2**: We apply the oracle reflection $U_f$ to the state $|s\rangle$.

![image4](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_step2.jpg)

Geometrically this corresponds to a reflection of the state $|s\rangle$ about $|s'\rangle$. This transformation means that the amplitude in front of the $|w\rangle$ state becomes negative, which in turn means that the average amplitude (indicated by a dashed line) has been lowered.

**Step 3**: We now apply an additional reflection ($U_s$) about the state $|s\rangle$: $U_s = 2|s\rangle\langle s| - \mathbb{1}$. This transformation maps the state to $U_s U_f| s \rangle$ and completes the transformation. 

![image5](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_step3.jpg)

Two reflections always correspond to a rotation. The transformation $U_s U_f$ rotates the initial state $|s\rangle$ closer towards the winner $|w\rangle$. The action of the reflection $U_s$ in the amplitude bar diagram can be understood as a reflection about the average amplitude. Since the average amplitude has been lowered by the first reflection, this transformation boosts the negative amplitude of $|w\rangle$ to roughly three times its original value, while it decreases the other amplitudes. We then go to **step 2** to repeat the application. This procedure will be repeated several times to zero in on the winner. 

After $t$ steps we will be in the state $|\psi_t\rangle$ where: $| \psi_t \rangle = (U_s U_f)^t  | s \rangle.$

How many times do we need to apply the rotation? It turns out that roughly $\sqrt{N}$ rotations suffice. This becomes clear when looking at the amplitudes of the state $| \psi \rangle$. We can see that the amplitude of $| w \rangle$ grows linearly with the number of applications $\sim t N^{-1/2}$. However, since we are dealing with amplitudes and not probabilities, the vector space's dimension enters as a square root. Therefore it is the amplitude, and not just the probability, that is being amplified in this procedure.

To calculate the number of rotations we need to know the size of the search space and the number of answers we are looking for. To get the optimal number of iterations $t$, we can follow the equation:     

$$
t = \lfloor\frac{\pi}{4}\sqrt{\frac{N}{m}}\rfloor 
$$

Where N is the size of the search space and m is the number of answers we want.


![image6](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_circuit_high_level.png)

&nbsp;

## 1.2 Grover Step by Step <a id='steps'></a>

Now that we went through how Grover's algorithm actually work, let's go a little bit in depth about the construction and different cases for each of its components.

##  1.2.1 Preparing the Search Space <a id='state-prep'></a>

The first step of Grover's algorithm is the initial state preparation. As we just mentioned, the search space is all possible values we need to search through to find the answer we want. For the examples in this textbook, our 'database' is comprised of all the possible computational basis states our qubits can be in. For example, if we have 3 qubits, our list is the states $|000\rangle, |001\rangle, \dots |111\rangle$ (i.e the states $|0\rangle \rightarrow |7\rangle$). So, in this case the size of our search space will be $N = 2^{3} = 8$.

In some cases, if we know the range within the search space where the answer is guaranteed to be, we can eliminate the redundant basis out of our search space to speed up the algorithm and decrease the size of the circuit. Generally speaking, we can prepare our state using any [symmetric states](gloss:ss), such as [GHZ-states](https://quantum-computing.ibm.com/composer/docs/iqx/example-circuits/ghz), [W-states](https://quantum-computing.ibm.com/composer/docs/iqx/example-circuits/w-state), or [Dicke-states](gloss:ds). 

For example, if we are trying to solve a problem with one answer using 4 qubits, and we prepare our state using the Hadamard gate (i.e. forming the Hilbert Space), N will be 16. But, if we know that the answer is within states when only one qubit has the value of 1 at any time, we can then use the W-state instead of the full Hilbert space to prepare our states. Doing that, decreased the size of the search space from 16 to 4 and the number of optimal iterations $t$ from 3 to 1.

## 1.2.2 Creating the Oracle <a id='oracle'></a>

The second and most important step of Grover’s algorithm is the oracle. Oracles add a negative phase to the solution states so they can standout from the rest and be measured. I.e. for any state $|x\rangle$ in the computational basis:

$$
U_\omega|x\rangle = \bigg\{
\begin{aligned}
\phantom{-}|x\rangle \quad \text{if} \; x \neq \omega \\
-|x\rangle \quad \text{if} \; x = \omega \\
\end{aligned}
$$

This oracle will be a diagonal matrix, where the entry that correspond to the marked item will have a negative phase. For example, if we have three qubits and $\omega = \text{101}$, our oracle will have the matrix:

$$
U_\omega = 
\begin{bmatrix}
1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 1 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 1 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 1 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & -1 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 \\
\end{bmatrix}
\begin{aligned}
\\
\\
\\
\\
\\
\\
\leftarrow \omega = \text{101}\\
\\
\\
\\
\end{aligned}
$$


What makes Grover’s algorithm so powerful is how easy it is to convert a problem to an oracle of this form. There are many computational problems in which it is difficult to _find_ a solution, but relatively easy to _verify_ a solution. For example, we can easily verify a solution to a [sudoku](https://en.wikipedia.org/wiki/Sudoku) by checking all the rules are satisfied. For these problems, we can create a function $f$ that takes a proposed solution $x$, and returns $f(x) = 0$ if $x$ is not a solution ($x \neq \omega$) and $f(x) = 1$ for a valid solution ($x = \omega$). Our oracle can then be described as:

$$
U_\omega|x\rangle = (-1)^{f(x)}|x\rangle
$$

and the oracle's matrix will be a diagonal matrix of the form:

$$
U_\omega = 
\begin{bmatrix}
(-1)^{f(0)} &   0         & \cdots &   0         \\
0           & (-1)^{f(1)} & \cdots &   0         \\
\vdots      &   0         & \ddots & \vdots      \\
0           &   0         & \cdots & (-1)^{f(2^n-1)} \\
\end{bmatrix}
$$


<!-- ::: q-block.reminder -->

### Detail

<details>
    <summary>Circuit Construction of a Grover Oracle</summary>
<p>
If we have our classical function $f(x)$, we can convert it to a reversible circuit of the form:
</p><p>
<img alt="A Classical Eeversible Oracle" src="https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_boolean_oracle.svg">
</p><p>
If we initialize the 'output' qubit in the state $|{-}\rangle$, the phase kickback effect turns this into a Grover oracle (similar to the workings of the Deutsch-Jozsa oracle):
</p><p>  
<img alt="Grover Oracle Constructed from a Classical Reversible Oracle" src="https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_phase_oracle.svg">
</p><p>
We then ignore the auxiliary ($|{-}\rangle$) qubit.
</p>
</details>

<!-- ::: -->



##  1.2.3 The Diffusion Operator <a id='diffusion'></a>

Finally, after the oracle has marked the correct answer by making it negative, the last step of Grover's algorithm which is the diffusion operator.

The construction of the diffusion operator depends on what we decide to use to prepare our initial states. Generally, the diffusion operator has the following construction.

![image7](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_diff.png)

For the next part of this chapter, we will create example oracles where we know $\omega$ beforehand, and not worry ourselves with whether these oracles are useful or not. At the end of the chapter, we will cover a short example where we create an oracle to solve a problem (sudoku) and a famous graph problem, the triangle finding problem.

## 2. Example: 2 Qubits <a id='2qubits'></a>

Let's first have a look at the case of Grover's algorithm for $N=4$ which is realized with 2 qubits. In this particular case, only <b>one rotation</b> is required to rotate the initial state $|s\rangle$ to the winner $|w\rangle$[3]:

1. Following  the above introduction, in the case $N=4$ we have 

   $$\theta = \arcsin \frac{1}{2} = \frac{\pi}{6}.$$

2. After $t$ steps, we have

   $$(U_s U_\omega)^t  | s \rangle = \sin \theta_t | \omega \rangle + \cos \theta_t | s' \rangle ,$$
   
   where
   
   $$\theta_t = (2t+1)\theta.$$

3. In order to obtain $| \omega \rangle$ we need $\theta_t = \frac{\pi}{2}$, which with $\theta=\frac{\pi}{6}$ inserted above results to $t=1$. This implies that after $t=1$ rotation the searched element is found.


We will now follow through an example using a specific oracle.

#### Oracle for $\lvert \omega \rangle = \lvert 11 \rangle$
Let's look at the case $\lvert w \rangle = \lvert 11 \rangle$. The oracle $U_\omega$ in this case acts as follows: 

$$U_\omega | s \rangle = U_\omega \frac{1}{2}\left( |00\rangle + |01\rangle + |10\rangle + |11\rangle \right) = \frac{1}{2}\left( |00\rangle + |01\rangle + |10\rangle - |11\rangle \right).$$

or:

$$
U_\omega = 
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & -1 \\
\end{bmatrix}
$$

which you may recognise as the controlled-Z gate. Thus, for this example, our oracle is simply the controlled-Z gate:

![image8](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_circuit_2qbuits_oracle_11.svg)

#### Reflection $U_s$

In order to complete the circuit we need to implement the additional reflection $U_s = 2|s\rangle\langle s| - \mathbb{1}$. Since this is a reflection about $|s\rangle$, we want to add a negative phase to every state orthogonal to $|s\rangle$. 

One way we can do this is to use the operation that transforms the state $|s\rangle \rightarrow |0\rangle$, which we already know is the Hadamard gate applied to each qubit:

$$H^{\otimes n}|s\rangle = |0\rangle$$

Then we apply a circuit that adds a negative phase to the states orthogonal to $|0\rangle$:

$$U_0 \frac{1}{2}\left( \lvert 00 \rangle + \lvert 01 \rangle + \lvert 10 \rangle + \lvert 11 \rangle \right) = \frac{1}{2}\left( \lvert 00 \rangle - \lvert 01 \rangle - \lvert 10 \rangle - \lvert 11 \rangle \right)$$

i.e. the signs of each state are flipped except for $\lvert 00 \rangle$. As can easily be verified, one way of implementing $U_0$ is the following circuit:

![Circuit for reflection around |0>](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_circuit_2qbuits_reflection_0.svg)

Finally, we do the operation that transforms the state $|0\rangle \rightarrow |s\rangle$ (the H-gate again):

$$H^{\otimes n}U_0 H^{\otimes n} = U_s$$

The complete circuit for $U_s$ looks like this:

![Circuit for reflection around |s>](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_circuit_2qbuits_reflection.svg)


#### Full Circuit for $\lvert w \rangle = |11\rangle$
Since in the particular case of $N=4$ only one rotation is required we can combine the above components to build the full circuit for Grover's algorithm for the case $\lvert w \rangle = |11\rangle$:

![image11](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_circuit_2qubits_full_11.svg)

### 2.1 Qiskit Implementation

We now implement Grover's algorithm for the above case of 2 qubits for $\lvert w \rangle = |11\rangle$.

In [ ]:
#initialization
import matplotlib.pyplot as plt        # plotting library (plot_histogram and our figures draw with it)
import numpy as np                     # numerical arrays and maths (np.sqrt, np.pi, np.arange, np.random)
import math                            # scalar maths of the standard library (math.comb, math.floor, math.asin, ...)
from IPython.display import display                      # show a figure (e.g. a circuit drawing) from the middle of a cell
def draw(qc, fold=-1, **kw): return qc.draw("mpl", style="clifford", fold=fold, **kw)   # every circuit picture: matplotlib drawer; fold=-1 = one long row, fold=N wraps after N columns


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `import numpy as np`, `import math` | NumPy for arrays (amplitude vectors such as `[1, -1]/np.sqrt(2)`), `math` for scalar functions such as `math.comb(n, k)`, used to size the triangle search space |
| `display(draw(qc))` | our one-line helper around `qc.draw("mpl", style="clifford")`; `display()` shows a figure from the middle of a cell, the `print("↑ ...")` after it is the caption |

</details>


In [ ]:
# importing Qiskit
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister   # circuits, and named registers of qubits / classical bits
# import basic plot tools
from qiskit.visualization import plot_histogram                         # bar chart of measurement counts
from qiskit.quantum_info import Statevector, Operator                   # exact state vector / unitary matrix of a circuit (checks by hand)
import qiskit; print(f"✓ Qiskit {qiskit.__version__} ready | TARGET = {TARGET}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumRegister(4, name='v')`, `ClassicalRegister` | named registers, used from Section 4 on: the sudoku circuit has variable (`v`), clause (`c`) and output (`out`) qubits, and the names appear on the drawings |
| `plot_histogram(counts)` | bar chart of a {bitstring: count} dict |
| `Statevector`, `Operator` | the exact state / unitary of a circuit, without sampling: used to check the oracles' sign patterns and in the In-depth analysis |

</details>


We start by preparing a quantum circuit with two qubits:

In [ ]:
n = 2                                  # number of qubits: N = 2**n = 4 items in the 'database'
grover_circuit = QuantumCircuit(n)     # QuantumCircuit(number_of_qubits): no classical bits yet (measure_all adds them later)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `n = 2` | 2 qubits = a 'database' of N = 2² = 4 items, labelled \|00⟩ … \|11⟩ |
| `QuantumCircuit(n)` | n qubits and no classical bits: `measure_all()` will add a register later |

</details>


Then we simply need to write out the commands for the circuit depicted above. First, we need to initialize the state $|s\rangle$. Let's create a general function (for any number of qubits) so we can use it again later:

In [ ]:
def initialize_s(qc, qubits):
    """Apply a H-gate to 'qubits' in qc"""
    for q in qubits:                   # loop over the list of qubit indices
        qc.h(q)                        # H|0⟩ = |+⟩ on every qubit builds |s⟩ = H^{⊗n}|0…0⟩, the uniform superposition
    return qc                          # the same (modified) circuit, so that calls can be chained


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `def initialize_s(qc, qubits):` | a reusable function; the `"""..."""` line under it is the docstring that `help(initialize_s)` prints |
| `qc.h(q)` in a loop | H\|0⟩ = (\|0⟩ + \|1⟩)/√2 on every qubit gives \|s⟩ = (1/√N) Σₓ \|x⟩: every item equally likely — 'we have no idea where the winner is' |
| `return qc` | the same circuit object, modified in place: `grover_circuit = initialize_s(grover_circuit, [0,1])` just re-binds the name |

</details>


In [ ]:
grover_circuit = initialize_s(grover_circuit, [0,1])   # |00⟩ -> |s⟩ = (|00⟩ + |01⟩ + |10⟩ + |11⟩)/2
display(draw(grover_circuit)); print("↑ state preparation: one H per qubit turns |00⟩ into the uniform superposition |s⟩ (amplitude 1/2 on each of the 4 items)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `initialize_s(grover_circuit, [0,1])` | H on qubits 0 and 1: amplitude 1/2 on each of \|00⟩, \|01⟩, \|10⟩, \|11⟩ |
| `display(draw(...)); print("↑ ...")` | the drawing and its caption: the same pattern for every circuit of the notebook |

</details>


Apply the Oracle for $|w\rangle = |11\rangle$. This oracle is specific to 2 qubits:

In [ ]:
grover_circuit.cz(0,1) # Oracle: CZ = diag(1, 1, 1, −1) flips the sign of |11⟩ only, the marked item w = 11
display(draw(grover_circuit)); print("↑ ... followed by the oracle U_ω for |w⟩ = |11⟩: a single CZ (a phase flip, invisible to a measurement on its own)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `grover_circuit.cz(0,1)` | the oracle for w = 11: CZ = diag(1, 1, 1, −1) multiplies the amplitude of \|11⟩ by −1 and does nothing else |
| a phase, not a probability | a measurement now would still give each outcome with probability 1/4: the sign becomes visible only after the diffuser turns it into amplitude |

</details>


<span id="general_diffuser"></span>
We now want to apply the diffuser ($U_s$). As with the circuit that initialises $|s\rangle$, we'll create a general diffuser (for any number of qubits) so we can use it later in other problems. 

In [ ]:
# Diffusion operator (U_s)
grover_circuit.h([0,1])      # H⊗H: |s⟩ -> |00⟩ (a list applies the gate to every qubit listed)
grover_circuit.z([0,1])      # Z⊗Z = diag(1, −1, −1, 1) ...
grover_circuit.cz(0,1)       # ... times CZ = diag(1, 1, 1, −1) gives diag(1, −1, −1, −1) = 2|00⟩⟨00| − I = U_0, the reflection about |00⟩
grover_circuit.h([0,1])      # H⊗H back: H⊗H · U_0 · H⊗H = 2|s⟩⟨s| − I = U_s, the reflection about |s⟩
display(draw(grover_circuit)); print("↑ the finished 2-qubit Grover circuit: H⊗H (|s⟩) | CZ (oracle U_ω) | H·Z·CZ·H (diffuser U_s) — one iteration")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `h([0,1])` … `h([0,1])` | conjugation by H⊗H turns a reflection about \|00⟩ into a reflection about \|s⟩ = H⊗H\|00⟩ |
| `z([0,1])` + `cz(0,1)` | Z⊗Z = diag(1, −1, −1, 1) times CZ = diag(1, 1, 1, −1) is diag(1, −1, −1, −1) = 2\|00⟩⟨00\| − I: the U₀ of the text, built from gates we already have |
| the whole circuit | one Grover iteration U_s U_ω applied to \|s⟩; for N = 4 one iteration is exactly enough (box below) |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**The geometric picture of one iteration.** Split the uniform state into its component along the winner $|w\rangle$ and the normalised rest $|s'\rangle$ (the $N-1$ losers; with $M$ winners, $|w\rangle$ is their normalised sum and $|s'\rangle$ the $N-M$ others):

$$|s\rangle = \sin\theta\,|w\rangle + \cos\theta\,|s'\rangle, \qquad \sin\theta = \langle w|s\rangle = \sqrt{M/N}.$$

The oracle $U_\omega = I - 2|w\rangle\langle w|$ reflects about $|s'\rangle$ (it flips the sign of the $|w\rangle$ component); the diffuser $U_s = 2|s\rangle\langle s| - I$ reflects about $|s\rangle$. Two reflections whose mirrors meet at angle $\theta$ make a rotation by $2\theta$, here towards $|w\rangle$. After $k$ iterations

$$(U_sU_\omega)^k|s\rangle = \sin\big((2k+1)\theta\big)\,|w\rangle + \cos\big((2k+1)\theta\big)\,|s'\rangle ,$$

so each winner has amplitude $\sin((2k+1)\theta)/\sqrt M$, each loser $\cos((2k+1)\theta)/\sqrt{N-M}$, and

$$P_{\rm success}(k) = \sin^2\big((2k+1)\theta\big).$$

**The same step in the amplitude bar chart** (Steps 2–3 of the introduction): the oracle flips the winner's bar, which lowers the average $\bar a$; the diffuser maps every amplitude $a \to 2\bar a - a$ (inversion about the average), so the negative winner comes back about three times taller while the others shrink. For small $\theta$ the winner's amplitude grows like $(2k+1)/\sqrt N$, linearly in $k$ — hence the $\sim\sqrt N$ iterations.

**This circuit.** $N = 4$, $M = 1$: $\sin\theta = 1/2$, $\theta = 30^\circ$; after $k = 1$ iteration the angle is $3\theta = 90^\circ$ and $P = \sin^2 90^\circ = 1$. The circuit drawn above should leave the qubits *exactly* in $|11\rangle$, which the state-vector check of Section 2.1.1 confirms.

</details>


This is our finished circuit.

### 2.1.1 Experiment with Simulators <a id='2qubits-simulation'></a>

Let's run the circuit in simulation. First, we can verify that we have the correct statevector:

In [ ]:
from qiskit import qpy                                  # QPY = Qiskit's binary file format for circuits

with open('grover_circuit.qpy', 'wb') as file:          # 'wb' = write binary; `with` closes the file at the end of the block
    qpy.dump(grover_circuit, file)                      # save the circuit as it is now (no measurements): reloaded in Section 2.1.2


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `from qiskit import qpy` | QPY: Qiskit's binary file format, which stores a circuit exactly (gates, parameters, registers) |
| `with open('grover_circuit.qpy', 'wb') as file:` | opens a file for **w**riting in **b**inary mode; `with` closes it automatically at the end of the block |
| `qpy.dump(grover_circuit, file)` | saves the 2-qubit circuit *before* any measurement (the file appears next to the notebook); Section 2.1.2 reloads it with `qpy.load` |

</details>


In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager   # transpiler, for the ideal Aer reference runs
from qiskit_aer import AerSimulator                                              # local ideal simulator: exact state vectors and noise-free references
from iqm.qiskit_iqm import transpile_to_IQM                                      # the IQM transpiler alone: native r (PRX) + cz on the chip's lattice
# runs on TARGET go through run_counts (imported in the TARGET cell): no Sampler in this edition


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `generate_preset_pass_manager`, `AerSimulator` | kept for the *ideal* references: the exact state vector below and the noise-free histograms the IQM runs are compared with |
| `transpile_to_IQM` | the IQM transpiler alone: rewrites a circuit in the native `r` (PRX) and `cz` gates on the chip's square lattice, without running it |
| no Sampler | runs on `TARGET` go through `run_counts` (imported in the `TARGET` cell), which returns a plain {bitstring: count} dict |

</details>


In [ ]:
sv_sim = AerSimulator()                              # ideal simulator; with save_statevector it returns the exact final state
sv_circuit = grover_circuit.copy()                   # a copy: the Aer-only save instruction must not end up in grover_circuit (it is run on hardware later)
sv_circuit.save_statevector()                        # store the state vector at this point of the circuit
result = sv_sim.run(sv_circuit).result()             # backend.run(circuit) -> job; .result() waits for it
statevec = result.get_statevector()                  # the 4 complex amplitudes, basis order |00⟩, |01⟩, |10⟩, |11⟩
from qiskit.visualization import array_to_latex      # pretty-print a vector/matrix as LaTeX
array_to_latex(statevec, prefix="|\\psi\\rangle =")  # the last expression is displayed: (0, 0, 0, 1) up to ~1e-16 rounding


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `sv_circuit = grover_circuit.copy()` | a copy: `save_statevector` is an Aer-only instruction that a hardware backend would reject, so it must not stay in `grover_circuit` (in the original it did) |
| `save_statevector()` + `.result().get_statevector()` | Aer stores the exact final state: no sampling, no noise |
| `array_to_latex(statevec, prefix=...)` | the 4 amplitudes (order \|00⟩, \|01⟩, \|10⟩, \|11⟩) as LaTeX: 0, 0, 0, 1 — numbers of order 10⁻¹⁶ are floating-point rounding |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**The 2-qubit case, amplitude by amplitude.** Basis order $|00\rangle, |01\rangle, |10\rangle, |11\rangle$.

| stage | amplitudes | average $\bar a$ |
|---|---|---|
| $H^{\otimes 2}\lvert 00\rangle = \lvert s\rangle$ | $(\tfrac12, \tfrac12, \tfrac12, \tfrac12)$ | $\tfrac12$ |
| oracle CZ | $(\tfrac12, \tfrac12, \tfrac12, -\tfrac12)$ | $\tfrac14$ |
| diffuser, $a \to 2\bar a - a$ | $(0, 0, 0, 1)$ | $\tfrac14$ |

The losers go to $2\cdot\tfrac14 - \tfrac12 = 0$, the winner to $2\cdot\tfrac14 + \tfrac12 = 1$. With matrices: $U_s = 2|s\rangle\langle s| - I$ has $-\tfrac12$ on the diagonal and $+\tfrac12$ everywhere else, and $U_s\,\tfrac12(1,1,1,-1)^T = (0,0,0,1)^T$. Geometrically: $\theta = 30^\circ$ and $3\theta = 90^\circ$ (box above).

The state vector printed by the cell is $(0, 0, 0, 1)$ up to $\sim10^{-16}$ rounding: the probability of $|11\rangle$ is exactly 1, so on an ideal simulator every shot gives `11`, and any other outcome in a run is noise.

</details>


As expected, the amplitude of every state that is not $|11\rangle$ is 0, this means we have a 100% chance of measuring $|11\rangle$:

In [ ]:
grover_circuit.measure_all()        # adds a barrier, a classical register named 'meas' (one bit per qubit) and the measurements


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `measure_all()` | a barrier, a classical register named `meas` with one bit per qubit, and one measurement per qubit |
| in place | modifies `grover_circuit` itself: the next cells transpile and run the measured circuit |

</details>


In [ ]:
# The circuit needs to be transpiled to the native gates of the IQM backend chosen by TARGET
backend = get_backend(TARGET)                                    # fake_apollo: a local noisy model of the 20-qubit Garnet chip
grover_isa_circuit = transpile_to_IQM(grover_circuit, backend=backend, optimization_level=3)   # native r (PRX) + cz, on physical qubits
print(f"transpiled for {backend.name}: {grover_isa_circuit.size()} native instructions, depth {grover_isa_circuit.depth()}, {dict(grover_isa_circuit.count_ops())}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `get_backend(TARGET)` | the Qiskit backend object for `TARGET`: `fake_apollo` = local noisy 20-qubit model (no token); `garnet:mock` = cloud pipeline check with RANDOM bitstrings; `garnet` / `emerald` = the real QPUs |
| `transpile_to_IQM(..., optimization_level=3)` | the 'ISA' circuit: H, Z and CZ rewritten in IQM's two native gates, `r` (PRX, a rotation about an axis in the xy-plane) and `cz`, on two coupled physical qubits |
| `size()`, `depth()`, `count_ops()` | what the chip executes: a few `r`, the two `cz` (oracle and diffuser) and the measurements — far below IQM Resonance's cap of 10 000 native instructions |

</details>


In [ ]:
counts = run_counts(grover_circuit, TARGET, shots=1024)          # transpile_to_IQM + backend.run + get_counts, in one call

def run_circuit(circuit, shots=1024):
    """Transpile + run on TARGET + counts in one call (used by the In-depth analysis at the end)."""
    return run_counts(circuit, TARGET, shots=shots)                # counts = plain dict {bitstring: count}

def to_native(circuit):
    """Rewrite a circuit in the native gates of IQM hardware, {r (PRX), cz}, on TARGET's qubit lattice: what the chip executes."""
    return transpile_to_IQM(circuit, backend=get_backend(TARGET), optimization_level=3)

print("counts:", counts)
plot_histogram(counts)                                           # the last expression of a cell is displayed: the bar chart


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `run_counts(grover_circuit, TARGET, shots=1024)` | the helper's full pipeline: `transpile_to_IQM` (native `r` and `cz`, routed on the square lattice) → `backend.run` → counts as a plain dict (no Sampler, no register names) |
| `run_circuit`, `to_native` | the same call in one function, and `transpile_to_IQM` alone (to inspect a circuit without running it) — both used by the In-depth analysis |
| `plot_histogram(counts)` | on `fake_apollo` about 87 % of the shots land on `11`, the rest leak to `01`, `10` and `00`: gate and readout noise, absent from the ideal probability 1 of the box above |

</details>


### 2.1.2 Experiment with Real Devices <a id='2qubits-device'></a>

We can run the circuit on IQM hardware as below. `TARGET` (set at the top) chooses the backend: with the default `"fake_apollo"` the run happens locally on a noisy model of the 20-qubit Garnet chip, with `"garnet"` / `"emerald"` on the real QPU via IQM Resonance (token + credits). There is no account to load, no queue to inspect and no least-busy selection: `run_counts` blocks until the counts are back.


In [ ]:
# IQM: no account object and no least-busy search — TARGET fixes the backend (the cloud targets read the token from IQM_TOKEN)
backend = get_backend(TARGET)                   # fake_apollo: built locally; cloud targets: a connection to IQM Resonance
couplers = {tuple(sorted(e)) for e in backend.coupling_map.get_edges()}   # set of unordered qubit pairs (an edge may be listed in both directions)
print(f"✓ Selected backend: {backend.name}")
print(f"  Qubits: {backend.num_qubits}")
print(f"  Coupling map: {len(couplers)} couplers (CZ-capable qubit pairs of the square lattice)")
print(f"  Native operations: {sorted(backend.operation_names)}")   # 'r' = PRX (phased X rotation), 'cz', 'measure', ...


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `get_backend(TARGET)` | a Qiskit `BackendV2`: `.name`, `.num_qubits`, `.coupling_map`, `.operation_names` — the same interface as an IBM device |
| `{tuple(sorted(e)) for e in ...}` | a set comprehension: (a, b) and (b, a) become the same pair, so we count physical couplers |
| `operation_names` | the native gates: `r` (PRX, rotation by an angle θ about an axis at angle φ in the xy-plane) and `cz` — every H, Z and multi-controlled gate must be rewritten in them |
| one cell instead of five | replaces the IBM edition's account loading, backend listing and least-busy selection: on IQM Resonance the backend is named explicitly by `TARGET` |

</details>


In [ ]:
display(draw(grover_circuit)); print("↑ the circuit as it is now: the 2-qubit Grover iteration followed by measure_all (barrier + one measurement per qubit)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `draw(grover_circuit)` | the circuit after the `measure_all()` of Section 2.1.1: barrier, then each qubit measured into the `meas` register |
| why look again | the next cell rebuilds the same circuit from the QPY file: the two drawings must agree |

</details>


In [ ]:
with open('grover_circuit.qpy', 'rb') as handle:       # 'rb' = read binary
    grover_circuit_list = qpy.load(handle)             # qpy.load returns a LIST of circuits (a file can hold several)
grover_circuit = grover_circuit_list[0]                # the one we saved: the 2-qubit Grover circuit without measurements
grover_circuit.measure_all()                           # add the measurements again
display(draw(grover_circuit)); print("↑ the circuit reloaded from grover_circuit.qpy, plus measure_all: identical to the one above")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qpy.load(handle)` | returns a list of circuits; `[0]` takes the first and only one |
| why reload | the save/load round trip: a circuit can be built in one session (or notebook) and executed in another |
| `measure_all()` | the file holds the measurement-free circuit saved in Section 2.1.1, so we measure again |

</details>


In [ ]:
# Transpile for the IQM backend: native r (PRX) + cz on the chip's square lattice (no run yet)
transpiled_grover_circuit = transpile_to_IQM(grover_circuit, backend=backend, optimization_level=3)
print(f"transpiled for {backend.name}: {transpiled_grover_circuit.size()} native instructions, depth {transpiled_grover_circuit.depth()}, {dict(transpiled_grover_circuit.count_ops())}")
display(draw(transpiled_grover_circuit, idle_wires=False)); print(f"↑ the circuit {backend.name} executes: every H and Z became r (PRX) rotations, the two CZ stay native cz on a coupled pair of physical qubits; idle_wires=False hides the other qubits of the chip")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `transpile_to_IQM(grover_circuit, backend=backend, optimization_level=3)` | the circuit the chip executes: native `r` (PRX) and `cz`, placed on two coupled physical qubits |
| `size()` | native instructions: IQM Resonance declines circuits above 10 000 — this one has a handful |
| `draw(..., idle_wires=False)` | only the qubits actually used; the labels are physical qubit numbers of the 20-qubit lattice |

</details>


In [ ]:
counts = run_counts(grover_circuit, TARGET, shots=1024)   # transpile_to_IQM + backend.run + counts: blocks until the result is back
print(f"\nIQM results on {backend.name} (TARGET = {TARGET!r}):")   # !r = repr(): the string with its quotes
print("Counts:", counts)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `run_counts(grover_circuit, TARGET, shots=1024)` | the full pipeline in one call; on Resonance the job is queued, executed and collected inside the call, so no session and no polling are needed |
| `TARGET` | `fake_apollo` = local noisy model (default); `garnet:mock` = cloud pipeline check (RANDOM bitstrings); `garnet` / `emerald` = real QPU (token + credits) |
| `counts` | a plain dict {bitstring: count}, the same type as `get_counts()` in the IBM edition |

</details>


In [ ]:
# Get the results from the computation
plot_histogram(counts)          # (the original plotted an undefined name, `answer`)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `plot_histogram(counts)` | the device(-like) run: most shots on `11`, the rest are errors (gates, readout, decoherence) — compare with the ideal 1024 of 1024 of Section 2.1.1 |
| `answer` → `counts` | the original referred to a variable no cell defined |

</details>


We confirm that in the majority of the cases the state $|11\rangle$ is measured. The other results are due to errors in the quantum computation. 

## 3. Example: 3 Qubits <a id='3qubits'></a>

We now go through the example of Grover's algorithm for 3 qubits with two marked states $\lvert101\rangle$ and $\lvert110\rangle$, following the implementation found in Reference [2]. The quantum circuit to solve the problem using a phase oracle is:

![image12](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_circuit_3qubits.png)

1. Apply Hadamard gates to $3$ qubits initialised to $\lvert000\rangle$ to create a uniform superposition:
   $$\lvert \psi_1 \rangle = \frac{1}{\sqrt{8}} \left( 
    \lvert000\rangle + \lvert001\rangle + \lvert010\rangle + \lvert011\rangle + 
    \lvert100\rangle + \lvert101\rangle + \lvert110\rangle + \lvert111\rangle \right)
   $$

2. Mark states $\lvert101\rangle$ and $\lvert110\rangle$ using a phase oracle:
   $$\lvert \psi_2 \rangle = \frac{1}{\sqrt{8}} \left( 
    \lvert000\rangle + \lvert001\rangle + \lvert010\rangle + \lvert011\rangle + 
    \lvert100\rangle - \lvert101\rangle - \lvert110\rangle + \lvert111\rangle \right)
   $$

3. Perform the reflection around the average amplitude:
   1. Apply Hadamard gates to the qubits
      $$
      \lvert \psi_{3a} \rangle = \frac{1}{2} \left( 
      \lvert000\rangle +\lvert011\rangle +\lvert100\rangle -\lvert111\rangle \right)
      $$
   
   2. Apply X gates to the qubits
      $$
      \lvert \psi_{3b} \rangle = \frac{1}{2} \left( 
      -\lvert000\rangle +\lvert011\rangle +\lvert100\rangle +\lvert111\rangle \right)
      $$
   
   3. Apply a doubly controlled Z gate between the 1, 2 (controls) and 3 (target) qubits
      $$
      \lvert \psi_{3c} \rangle = \frac{1}{2} \left( 
      -\lvert000\rangle +\lvert011\rangle +\lvert100\rangle -\lvert111\rangle \right)
      $$
   
   4. Apply X gates to the qubits
      $$
      \lvert \psi_{3d} \rangle = \frac{1}{2} \left( 
      -\lvert000\rangle +\lvert011\rangle +\lvert100\rangle -\lvert111\rangle \right)
      $$
   
   5. Apply Hadamard gates to the qubits
      $$
      \lvert \psi_{3e} \rangle = \frac{1}{\sqrt{2}} \left( 
      -\lvert101\rangle -\lvert110\rangle \right)
      $$

4. Measure the $3$ qubits to retrieve states $\lvert101\rangle$ and $\lvert110\rangle$

Note that since there are 2 solutions and 8 possibilities, we will only need to run one iteration (steps 2 & 3).

### 3.1 Qiskit Implementation <a id='3qubit-implementation'></a>

We now implement Grover's algorithm for the above [example](#3qubits) for $3$-qubits and searching for two marked states $\lvert101\rangle$ and $\lvert110\rangle$. **Note:** Remember that Qiskit orders it's qubits the opposite way round to this resource, so the circuit drawn will appear flipped about the horizontal.

We create a phase oracle that will mark states $\lvert101\rangle$ and $\lvert110\rangle$ as the results (step 1).

In [ ]:
qc = QuantumCircuit(3)
qc.cz(0, 2)                          # −1 when q0 = q2 = 1: states |101⟩ and |111⟩ (bitstrings are written q2 q1 q0)
qc.cz(1, 2)                          # −1 when q1 = q2 = 1: states |110⟩ and |111⟩ -> |111⟩ gets (−1)² = +1
oracle_ex3 = qc.to_gate()            # turn the circuit into a single gate (a box in later drawings)
oracle_ex3.name = r"U$_\omega$"      # the box label; r"..." = raw string, so the backslash of the LaTeX \omega is kept as is
display(draw(qc)); print("↑ the phase oracle for |101⟩ and |110⟩: two CZ gates sharing qubit 2")
print("diagonal of the oracle (states 000 … 111):", np.round(np.diag(Operator(qc).data).real).astype(int))   # Operator(qc).data = the 8×8 unitary


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| two `cz` sharing qubit 2 | each CZ flips the sign when both its qubits are 1; \|111⟩ is flipped twice, so only \|101⟩ and \|110⟩ end up with −1 |
| `np.diag(Operator(qc).data)` | the diagonal of the 8×8 unitary: `[1 1 1 1 1 -1 -1 1]`, −1 at positions 5 = 101 and 6 = 110 |
| `qc.to_gate()`, `.name = r"U$_\omega$"` | packs the circuit as one gate with a LaTeX label; the raw string avoids the `SyntaxWarning` the original plain string gave for `\o` |

</details>


In the last section, we used a diffuser specific to 2 qubits, in the cell below we will create a general diffuser for any number of qubits.


<!-- ::: q-block.reminder -->

### Detail

<details>
<summary>Creating a General Diffuser</summary>
    
Remember that we can create $U_s$ from $U_0$:

$$ U_s = H^{\otimes n} U_0 H^{\otimes n} $$

And a multi-controlled-Z gate ($MCZ$) inverts the phase of the state $|11\dots 1\rangle$:

$$
MCZ = 
\begin{bmatrix}
 1 & 0 & 0 & \cdots & 0 \\
 0 & 1 & 0 & \cdots & 0 \\
 \vdots & \vdots & \vdots & \ddots & \vdots \\
 0 & 0 & 0 & \cdots & -1 \\
\end{bmatrix}
\begin{aligned}
\\
\\
\\
\leftarrow \text{Add negative phase to} \; |11\dots 1\rangle\\
\end{aligned}
$$

Applying an X-gate to each qubit performs the transformation:

$$
\begin{aligned}
|00\dots 0\rangle & \rightarrow |11\dots 1\rangle\\
|11\dots 1\rangle & \rightarrow |00\dots 0\rangle
\end{aligned}
$$

So:

$$ U_0 = - X^{\otimes n} (MCZ) X^{\otimes n} $$

Using these properties together, we can create $U_s$ using H-gates, X-gates, and a single multi-controlled-Z gate:

$$ U_s = - H^{\otimes n} U_0 H^{\otimes n} = H^{\otimes n} X^{\otimes n} (MCZ) X^{\otimes n} H^{\otimes n} $$
    
Note that we can ignore the global phase of -1.

</details>

<!-- ::: -->

In [ ]:
def diffuser(nqubits):
    qc = QuantumCircuit(nqubits)
    # Apply transformation |s> -> |00..0> (H-gates)
    for qubit in range(nqubits):
        qc.h(qubit)
    # Apply transformation |00..0> -> |11..1> (X-gates)
    for qubit in range(nqubits):
        qc.x(qubit)
    # Do multi-controlled-Z gate
    qc.h(nqubits-1)                                  # H on the target turns the multi-controlled X below into a multi-controlled Z
    qc.mcx(list(range(nqubits-1)), (nqubits-1))  # multi-controlled-toffoli: X on the last qubit if all the others are 1
    qc.h(nqubits-1)
    # Apply transformation |11..1> -> |00..0>
    for qubit in range(nqubits):
        qc.x(qubit)
    # Apply transformation |00..0> -> |s>
    for qubit in range(nqubits):
        qc.h(qubit)
    # We will return the diffuser as a gate
    U_s = qc.to_gate()
    U_s.name = "U$_s$"
    return U_s

display(draw(diffuser(3).definition)); print("↑ the general diffuser for 3 qubits: H⊗3 · X⊗3 · (H·CCX·H = CCZ on the last qubit) · X⊗3 · H⊗3 = −(2|s⟩⟨s| − I)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `h` · `mcx` · `h` on the last qubit | a multi-controlled Z: −1 on \|1…1⟩ only (H X H = Z on the target) |
| X⊗n · MCZ · X⊗n | moves that −1 to \|0…0⟩: I − 2\|0…0⟩⟨0…0\| = −U₀ |
| H⊗n · … · H⊗n | turns it into −(2\|s⟩⟨s\| − I) = −U_s; the global −1 is unobservable (details box above) |
| `.to_gate()`, `.definition` | the diffuser is returned as a gate (a box); `.definition` gives back the circuit inside it, for the drawing |

</details>


We'll now put the pieces together, with the creation of a uniform superposition at the start of the circuit and a measurement at the end. Note that since there are 2 solutions and 8 possibilities, we will only need to run one iteration. 

*(A scratch snippet left by the author, kept as text: it would draw a single Toffoli on 3 qubits; `nqubits` must be set first.)*

```python
# nqubits = 3
qc = QuantumCircuit(nqubits)
qc.ccx([0],[1],[2])  # multi-controlled-toffoli
qc.draw('mpl')
```


In [ ]:
n = 3
grover_circuit = QuantumCircuit(n)
grover_circuit = initialize_s(grover_circuit, [0,1,2])   # |s⟩ on 3 qubits: amplitude 1/√8 on each of the 8 items
grover_circuit.append(oracle_ex3, [0,1,2])               # append(gate, qubits): the oracle box, marking |101⟩ and |110⟩
grover_circuit.append(diffuser(n), [0,1,2])              # the diffuser box U_s: one iteration suffices for M = 2 of N = 8
grover_circuit.measure_all()                             # barrier + 'meas' register + measurements


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `append(gate, qubits)` | inserts the gate as a box on the listed qubits (`compose` would copy the gates of a circuit instead) |
| one iteration | N = 8, M = 2: sin θ = √(2/8) = 1/2, θ = 30°, and after one iteration 3θ = 90°: only the two marked states remain (table in the box after the sudoku) |
| `grover_circuit` re-used | the name now refers to the 3-qubit circuit |

</details>


In [ ]:
display(draw(grover_circuit)); print("↑ the 3-qubit search at the level of boxes: H⊗3 | oracle U_ω | diffuser U_s | measurement")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `draw(grover_circuit)` | the boxes stay closed: this is the algorithm as a sequence of steps |
| the boxes' labels | the `.name` strings set above, rendered as LaTeX by the mpl drawer |

</details>


In [ ]:
display(draw(grover_circuit.decompose())); print(f"↑ one level of decompose(): the boxes opened into their gates — {dict(grover_circuit.decompose().count_ops())}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `grover_circuit.decompose()` | replaces every box by its definition, one level deep: the two CZ of the oracle and the H/X/MCX of the diffuser appear |
| `count_ops()` in the caption | the gate counts of the opened circuit |

</details>


### 3.1.1 Experiment with Simulators  <a id='3qubits-simulation'></a>

We can run the above circuit on the backend chosen by `TARGET` (with the default `fake_apollo`, a noisy simulator of the Garnet chip).


In [ ]:
counts = run_counts(grover_circuit, TARGET, shots=1024)          # the 3-qubit search on TARGET (transpile_to_IQM opens the boxes)
print("counts:", counts)
plot_histogram(counts)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `run_counts(grover_circuit, TARGET, shots=1024)` | transpilation to `r` + `cz` (the boxes are unrolled, the Toffoli of the diffuser becomes several `cz`) and the run on `TARGET` |
| `plot_histogram(counts)` | on `fake_apollo` `101` and `110` keep roughly 35–37 % each (ideal: 50 % each and nothing elsewhere); the other six outcomes collect the noise of about ten `cz` gates |

</details>


As we can see, the algorithm discovers our marked states $\lvert101\rangle$ and $\lvert110\rangle$.

### 3.1.2 Experiment with Real Devices  <a id='3qubits-device'></a>

We can run the circuit on the IQM backend chosen by `TARGET` as below.


In [ ]:
backend = get_backend(TARGET)                   # the same backend as in Section 2.1.2 (TARGET fixes it)
couplers = {tuple(sorted(e)) for e in backend.coupling_map.get_edges()}
print(f"✓ Selected backend: {backend.name} — {backend.num_qubits} qubits, {len(couplers)} couplers")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `get_backend(TARGET)` | the backend chosen at the top; no listing and no least-busy selection on IQM Resonance |
| `couplers` | the CZ-capable qubit pairs: a Toffoli needs its three qubits pairwise coupled, which the square lattice does not offer, so the router adds swaps |

</details>


In [ ]:
transpiled_grover_circuit = transpile_to_IQM(grover_circuit, backend=backend, optimization_level=3)   # native r + cz on the lattice
print(f"transpiled for {backend.name}: {transpiled_grover_circuit.size()} native instructions, depth {transpiled_grover_circuit.depth()}, {dict(transpiled_grover_circuit.count_ops())}")
display(draw(transpiled_grover_circuit, idle_wires=False)); print(f"↑ the 3-qubit search as {backend.name} runs it: the oracle's two CZ and the diffuser's Toffoli became native cz gates (plus routing) dressed with r rotations")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `transpile_to_IQM(...)` | the circuit in `r` (PRX) and `cz` on physical qubits of the square lattice |
| `count_ops()['cz']` | the two-qubit gates, the noisiest operations: the Toffoli alone needs about six, plus the routing |
| `size()` | far below the 10 000-instruction cap of IQM Resonance |

</details>


In [ ]:
counts = run_counts(grover_circuit, TARGET, shots=1024)   # transpile_to_IQM + backend.run + counts
print(f"\nIQM results on {backend.name} (TARGET = {TARGET!r}):")
print("Counts:", counts)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `run_counts(grover_circuit, TARGET, shots=1024)` | the same pipeline as in Section 2.1.2, now for the 3-qubit search |
| `counts` | plain dict {bitstring: count}; on `fake_apollo` the noise of the `cz` gates spreads shots over all 8 outcomes |

</details>


In [ ]:
# Get the results from the computation
plot_histogram(counts)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `plot_histogram(counts)` | `101` and `110` should still stand out; everything else is noise — compare with the ideal histogram of Section 3.1.1 |
| deeper circuit, more noise | the 3-qubit search has several times the two-qubit gates of the 2-qubit one, and it shows |

</details>


As we can (hopefully) see, there is a higher chance of measuring $\lvert101\rangle$ and $\lvert110\rangle$. The other results are due to errors in the quantum computation. 

## 4. Problems <a id='problems'></a>

The function `grover_problem_oracle` below takes a number of qubits (`n`), and a `variant` and returns an n-qubit oracle. The function will always return the same oracle for the same `n` and `variant`. You can see the solutions to each oracle by setting `print_solutions = True` when calling `grover_problem_oracle`.

In [ ]:
# The original imported grover_problem_oracle from the qiskit_textbook package, which is retired and no longer
# installs with Qiskit 2.x: the function below is a self-contained stand-in with the same interface.
def grover_problem_oracle(n, variant=0, print_solutions=False):
    """An n-qubit phase oracle (a gate) with 'hidden' solutions: the same n and variant always give the same oracle.
    Even variants have 1 solution, odd variants 2 (as Problems 1 and 2 below assume for n = 4)."""
    rng = np.random.default_rng(100 * n + variant)               # seeded random generator: reproducible hidden solutions
    n_solutions = 1 if variant % 2 == 0 else 2
    solutions = [format(int(x), f"0{n}b") for x in rng.choice(2**n, size=n_solutions, replace=False)]   # distinct n-bit strings
    if print_solutions:
        print("Solutions:", solutions)
    qc = QuantumCircuit(n)
    for s in solutions:                                          # one phase flip per solution
        zeros = [q for q in range(n) if s[n - 1 - q] == "0"]     # qubits that are 0 in s (the bitstring is written q_{n-1} … q_0)
        if zeros: qc.x(zeros)                                    # relabel s -> 11…1
        qc.h(n - 1); qc.mcx(list(range(n - 1)), n - 1); qc.h(n - 1)   # multi-controlled Z: −1 on |11…1⟩ only
        if zeros: qc.x(zeros)                                    # undo the relabelling: net effect −1 on |s⟩
    oracle = qc.to_gate()
    oracle.name = "Oracle"
    return oracle

## Example Usage
n = 4
oracle = grover_problem_oracle(n, variant=1)  # variant 1 of the oracle (2 solutions), with n qubits
qc = QuantumCircuit(n)
qc.append(oracle, [0,1,2,3])
display(draw(qc)); print("↑ a problem oracle is a black box: its solutions are hidden (print_solutions=True reveals them)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `grover_problem_oracle(n, variant)` | replaces the import from the retired `qiskit_textbook` package: same call, a phase oracle with hidden solutions (1 for even, 2 for odd variants) |
| `np.random.default_rng(100 * n + variant)` | a seeded generator: the 'random' solutions are the same at every run, so the problems below have fixed answers |
| `X` · (`H`·`mcx`·`H`) · `X` | the phase flip of one bitstring s: relabel s to 11…1, apply a multi-controlled Z, undo the relabelling |
| `qc.append(oracle, [0,1,2,3])` | the oracle as a single box: you are not supposed to look inside (`.decompose()` would reveal the solutions) |

</details>


1. `grover_problem_oracle(4, variant=2)` uses 4 qubits and has 1 solution.    
   a. How many iterations do we need to have a > 90% chance of measuring this solution?    
   b. Use Grover's algorithm to find this solution state.
   c. What happens if we apply more iterations than the number we calculated in problem 1a above? Why?

2. With 2 solutions and 4 qubits, how many iterations do we need for a >90% chance of measuring a solution? Test your answer using the oracle `grover_problem_oracle(4, variant=1)` (which has two solutions).

3. Create a function, `grover_solver(oracle, iterations)` that takes as input:
    - A Grover oracle as a gate (`oracle`)
    - An integer number of iterations (`iterations`)
    
   and returns a `QuantumCircuit` that performs Grover's algorithm on the '`oracle`' gate, with '`iterations`' iterations.

## 4.1 Solving Sudoku using Grover's Algorithm <a id="sudoku"></a>

The oracles used throughout this chapter so far have been created with prior knowledge of their solutions. We will now solve a simple problem using Grover's algorithm, for which we do not necessarily know the solution beforehand. Our problem is a 2×2 binary sudoku, which in our case has two simple rules:

- No column may contain the same value twice
- No row may contain the same value twice

If we assign each square in our sudoku to a variable like so:

![2×2 binary sudoku, with each square allocated to a different variable](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/binary_sudoku.png)

we want our circuit to output a solution to this sudoku.

Note that, while this approach of using Grover's algorithm to solve this problem is not practical (you can probably find the solution in your head!), the purpose of this example is to demonstrate the conversion of classical [decision problems](https://en.wikipedia.org/wiki/Decision_problem) into oracles for Grover's algorithm.

### 4.1.1 Turning the Problem into a Circuit

We want to create an oracle that will help us solve this problem, and we will start by creating a circuit that identifies a correct solution. Similar to how we created a classical adder using quantum circuits in [_The Atoms of Computation_](/course/ch-states/the-atoms-of-computation), we simply need to create a _classical_ function on a quantum circuit that checks whether the state of our variable bits is a valid solution.

Since we need to check down both columns and across both rows, there are 4 conditions we need to check:

```
v0 ≠ v1   # check along top row
v2 ≠ v3   # check along bottom row
v0 ≠ v2   # check down left column
v1 ≠ v3   # check down right column
```

Remember we are comparing classical (computational basis) states. For convenience, we can compile this set of comparisons into a list of clauses:

In [ ]:
clause_list = [[0,1],      # v0 ≠ v1: top row
               [0,2],      # v0 ≠ v2: left column
               [1,3],      # v1 ≠ v3: right column
               [2,3]]      # v2 ≠ v3: bottom row


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `clause_list` | a list of lists: each pair of cell indices must hold different values — the four sudoku rules |
| `[0,1]` | v0 ≠ v1 (top row); the order of the clauses fixes which clause qubit c0 … c3 stores which check |

</details>


We will assign the value of each variable to a bit in our circuit. To check these clauses computationally, we will use the `XOR` gate (we came across this in the atoms of computation).

In [ ]:
def XOR(qc, a, b, output):
    qc.cx(a, output)           # output ^= a   (CNOT = classical XOR into the target)
    qc.cx(b, output)           # output ^= b   -> output = a ⊕ b if it started at 0


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.cx(a, output)` | CNOT on basis states: output → output ⊕ a |
| two CNOTs | output → output ⊕ a ⊕ b: starting from 0, `output` = 1 exactly when a ≠ b — a reversible clause check |

</details>


Convince yourself that the `output0` bit in the circuit below will only be flipped if `input0 ≠ input1`:

In [ ]:
# We will use separate registers to name the bits
in_qubits = QuantumRegister(2, name='input')         # a 2-qubit register called 'input'
out_qubit = QuantumRegister(1, name='output')
qc = QuantumCircuit(in_qubits, out_qubit)            # a circuit built from registers (the drawing shows their names)
XOR(qc, in_qubits[0], in_qubits[1], out_qubit)       # register[i] = one qubit of the register
display(draw(qc)); print("↑ the XOR gadget: output0 is flipped once for each input that is 1, i.e. output0 = input0 ⊕ input1")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumRegister(2, name='input')` | a named group of qubits; `in_qubits[0]` is its first qubit |
| `QuantumCircuit(in_qubits, out_qubit)` | a circuit made of registers instead of a bare qubit count |
| the drawing | two CNOTs onto `output`: \|00⟩ and \|11⟩ leave it at 0, \|01⟩ and \|10⟩ set it to 1 |

</details>


This circuit checks whether `input0 == input1` and stores the output to `output0`. To check each clause, we repeat this circuit for each pairing in `clause_list` and store the output to a new bit:

In [ ]:
# Create separate registers to name bits
var_qubits = QuantumRegister(4, name='v')  # variable bits
clause_qubits = QuantumRegister(4, name='c')  # bits to store clause-checks

# Create quantum circuit
qc = QuantumCircuit(var_qubits, clause_qubits)

# Use XOR gate to check each clause
i = 0
for clause in clause_list:
    XOR(qc, clause[0], clause[1], clause_qubits[i])   # clause = [a, b]: c_i = v_a ⊕ v_b (integers 0–3 address the first register, v)
    i += 1

display(draw(qc)); print("↑ the four clause checks: c0 = v0⊕v1, c1 = v0⊕v2, c2 = v1⊕v3, c3 = v2⊕v3 (8 CNOTs)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumRegister(4, name='v')`, `(4, name='c')` | 4 variable qubits (the sudoku cells) and 4 clause qubits (one per rule) |
| `XOR(qc, clause[0], clause[1], clause_qubits[i])` | an integer index like `0` means circuit qubit 0 = v0; `clause_qubits[i]` is the i-th qubit of register c |
| `i = 0` … `i += 1` | a manual counter (`for i, clause in enumerate(clause_list)` would do the same) |

</details>


The final state of the bits `c0, c1, c2, c3` will only all be `1` in the case that the assignments of `v0, v1, v2, v3` are a solution to the sudoku. To complete our checking circuit, we want a single bit to be `1` if (and only if) all the clauses are satisfied, this way we can look at just one bit to see if our assignment is a solution. We can do this using a multi-controlled-Toffoli-gate:

In [ ]:
# Create separate registers to name bits
var_qubits = QuantumRegister(4, name='v')
clause_qubits = QuantumRegister(4, name='c')
output_qubit = QuantumRegister(1, name='out')
qc = QuantumCircuit(var_qubits, clause_qubits, output_qubit)

# Compute clauses
i = 0
for clause in clause_list:
    XOR(qc, clause[0], clause[1], clause_qubits[i])
    i += 1

# Flip 'output' bit if all clauses are satisfied
qc.mcx(clause_qubits, output_qubit)        # 4-controlled X (the original qc.mct was removed in Qiskit 1.0): out ^= c0 c1 c2 c3

display(draw(qc)); print("↑ the checking circuit: clause checks, then a 4-controlled X that sets 'out' only if all four clauses hold")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.mcx(clause_qubits, output_qubit)` | multi-controlled X: flips `out` only if all four clause qubits are 1 (an AND of the clauses); replaces `mct`, removed in Qiskit 1.0 |
| `output_qubit` | a third register with one qubit, named `out` |
| the whole circuit | a reversible classical verifier: on \|x⟩\|0000⟩\|0⟩ it leaves out = f(x), 1 for a valid sudoku |

</details>


The circuit above takes as input an initial assignment of the bits `v0`, `v1`, `v2` and `v3`, and all other bits should be initialised to `0`. After running the circuit, the state of the `out0` bit tells us if this assignment is a solution or not; `out0 = 0` means the assignment _is not_ a solution, and `out0 = 1` means the assignment _is_ a solution.

**Important:** Before you continue, it is important you fully understand this circuit and are convinced it works as stated in the paragraph above.

### 4.1.2 Uncomputing, and Completing the Oracle

We can now turn this checking circuit into a Grover oracle using [phase kickback](/course/ch-gates/phase-kickback). To recap, we have 3 registers: 
- One register which stores our sudoku variables (we'll say $x = v_3, v_2, v_1, v_0$)
- One register that stores our clauses (this starts in the state $|0000\rangle$ which we'll abbreviate to $|0\rangle$)
- And one qubit ($|\text{out}_0\rangle$) that we've been using to store the output of our checking circuit. 

To create an oracle, we need our circuit ($U_\omega$) to perform the transformation:

$$
U_\omega|x\rangle|0\rangle|\text{out}_0\rangle = |x\rangle|0\rangle|\text{out}_0\oplus f(x)\rangle
$$

If we set the `out0` qubit to the superposition state $|{-}\rangle$ we have:

$$
\begin{aligned}
U_\omega|x\rangle|0\rangle|{-}\rangle 
&= U_\omega|x\rangle|0\rangle\otimes\tfrac{1}{\sqrt{2}}(|0\rangle - |1\rangle)\\
&= |x\rangle|0\rangle\otimes\tfrac{1}{\sqrt{2}}(|0\oplus f(x)\rangle - |1\oplus f(x)\rangle)
\end{aligned}
$$

If $f(x) = 0$, then we have the state:

$$
\begin{aligned}
&= |x\rangle|0\rangle\otimes \tfrac{1}{\sqrt{2}}(|0\rangle - |1\rangle)\\
&= |x\rangle|0\rangle|-\rangle\\
\end{aligned}
$$


(i.e. no change). But if $f(x) = 1$ (i.e. $x = \omega$), we introduce a negative phase to the $|{-}\rangle$ qubit:

$$
\begin{aligned}
&= \phantom{-}|x\rangle|0\rangle\otimes\tfrac{1}{\sqrt{2}}(|1\rangle - |0\rangle)\\
&= \phantom{-}|x\rangle|0\rangle\otimes -\tfrac{1}{\sqrt{2}}(|0\rangle - |1\rangle)\\
&= -|x\rangle|0\rangle|-\rangle\\
\end{aligned}
$$

This is a functioning oracle that uses two auxiliary registers in the state $|0\rangle|{-}\rangle$:

$$
U_\omega|x\rangle|0\rangle|{-}\rangle = \Bigg\{
\begin{aligned}
\phantom{-}|x\rangle|0\rangle|-\rangle \quad \text{for} \; x \neq \omega \\
-|x\rangle|0\rangle|-\rangle \quad \text{for} \; x = \omega \\
\end{aligned}
$$

To adapt our checking circuit into a Grover oracle, we need to guarantee the bits in the second register (`c`) are always returned to the state $|0000\rangle$ after the computation. To do this, we simply repeat the part of the circuit that computes the clauses which guarantees `c0 = c1 = c2 = c3 = 0` after our circuit has run. We call this step _'uncomputation'_.

In [ ]:
var_qubits = QuantumRegister(4, name='v')
clause_qubits = QuantumRegister(4, name='c')
output_qubit = QuantumRegister(1, name='out')
cbits = ClassicalRegister(4, name='cbits')                # 4 classical bits for the final measurement of the variables
qc = QuantumCircuit(var_qubits, clause_qubits, output_qubit, cbits)

def sudoku_oracle(qc, clause_list, clause_qubits):
    # Compute clauses
    i = 0
    for clause in clause_list:
        XOR(qc, clause[0], clause[1], clause_qubits[i])
        i += 1

    # Flip 'output' bit if all clauses are satisfied
    qc.mcx(clause_qubits, output_qubit)                   # (uses the global output_qubit register)

    # Uncompute clauses to reset clause-checking bits to 0
    i = 0
    for clause in clause_list:
        XOR(qc, clause[0], clause[1], clause_qubits[i])   # XOR twice = identity: c_i back to 0
        i += 1

sudoku_oracle(qc, clause_list, clause_qubits)
display(draw(qc)); print("↑ the sudoku oracle: compute the 4 clauses (8 CNOTs) | 4-controlled X on 'out' | uncompute the clauses (the same 8 CNOTs)")

# Check (added): feed every assignment x = v3 v2 v1 v0 as a basis state, apply the oracle, read 'out' and the clause bits
marked = []
for x in range(16):
    test = QuantumCircuit(var_qubits, clause_qubits, output_qubit)
    for q in range(4):
        if (x >> q) & 1:                                  # bit q of the integer x
            test.x(var_qubits[q])                         # |0000⟩ -> |x⟩
    sudoku_oracle(test, clause_list, clause_qubits)
    final = Statevector(test).probabilities_dict()        # basis state in, basis state out: a single key with probability 1
    bits = max(final, key=final.get)                      # the bitstring 'out c3 c2 c1 c0 v3 v2 v1 v0'
    assert bits[1:5] == "0000"                            # uncomputation worked: the clause register is back to |0000⟩
    if bits[0] == "1":                                    # 'out' was flipped: x satisfies every clause
        marked.append(format(x, "04b"))
print("assignments v3v2v1v0 that flip 'out':", marked)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `ClassicalRegister(4, name='cbits')` | 4 classical bits, used at the end to measure the variables only |
| `sudoku_oracle(qc, ...)` | compute → flip → uncompute: the second loop repeats the XORs, and since XOR twice is the identity the clause qubits return to 0 |
| check loop (added) | for each of the 16 inputs: prepare \|x⟩ with X gates, run the oracle, read the single output basis state from `Statevector` |
| `max(final, key=final.get)` | the key with the largest value of the dict, i.e. the output basis state |
| `assert bits[1:5] == "0000"` | stops with an error if a clause qubit were left at 1; it passes for all 16 inputs |
| printed list | `['0110', '1001']`: the only two valid 2×2 binary sudokus (box below) |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**The sudoku oracle is a phase flip on the satisfying assignments.** Write $x = v_3v_2v_1v_0$; clause qubit $i$ computes $c_i = v_a\oplus v_b$ for the pair $(a, b)$ of clause $i$, and the checking function is the AND of the four clauses:

$$f(x) = c_0\,c_1\,c_2\,c_3 = (v_0\oplus v_1)(v_0\oplus v_2)(v_1\oplus v_3)(v_2\oplus v_3).$$

It is 1 iff $v_1 = v_2 = \bar v_0$ and $v_3 = v_0$: $x \in \{0110,\ 1001\}$, the list printed by the cell.

*Compute – flip – uncompute.* On a basis state the circuit does

$$|x\rangle|0000\rangle|o\rangle \xrightarrow{\text{XORs}} |x\rangle|c(x)\rangle|o\rangle \xrightarrow{C^4X} |x\rangle|c(x)\rangle|o\oplus f(x)\rangle \xrightarrow{\text{XORs}} |x\rangle|0000\rangle|o\oplus f(x)\rangle .$$

*Phase kickback.* With the output qubit in $|{-}\rangle$, flipping it gives $X|{-}\rangle = -|{-}\rangle$, so

$$|x\rangle|0000\rangle|{-}\rangle \;\to\; (-1)^{f(x)}\,|x\rangle|0000\rangle|{-}\rangle ,$$

and on the variable register alone the oracle is $U_\omega = \mathrm{diag}\big((-1)^{f(0)}, \dots, (-1)^{f(15)}\big)$, with $-1$ at positions $6 = 0110$ and $9 = 1001$ and $+1$ elsewhere — exactly the kind of oracle of Section 1.2.2, built without knowing the solutions in advance.

*Why uncompute?* Without the second round of XORs the clause register would stay in $|c(x)\rangle$, different for different $x$: the variables would be entangled with it, and the diffuser, which acts on the variables only, could no longer make the 16 branches interfere. The `assert` of the cell checks that the clause register is back to $|0000\rangle$ for every input.

</details>


In summary, the circuit above performs:

$$
U_\omega|x\rangle|0\rangle|\text{out}_0\rangle = \Bigg\{
\begin{aligned}
|x\rangle|0\rangle|\text{out}_0\rangle \quad \text{for} \; x \neq \omega \\
|x\rangle|0\rangle\otimes X|\text{out}_0\rangle \quad \text{for} \; x = \omega \\
\end{aligned}
$$

and if the initial state of $|\text{out}_0\rangle = |{-}\rangle$,:

$$
U_\omega|x\rangle|0\rangle|{-}\rangle = \Bigg\{
\begin{aligned}
\phantom{-}|x\rangle|0\rangle|-\rangle \quad \text{for} \; x \neq \omega \\
-|x\rangle|0\rangle|-\rangle \quad \text{for} \; x = \omega \\
\end{aligned}
$$

### 4.1.3 The Full Algorithm

All that's left to do now is to put all these components together.

In [ ]:
var_qubits = QuantumRegister(4, name='v')
clause_qubits = QuantumRegister(4, name='c')
output_qubit = QuantumRegister(1, name='out')
cbits = ClassicalRegister(4, name='cbits')
qc = QuantumCircuit(var_qubits, clause_qubits, output_qubit, cbits)

# Initialize 'out0' in state |->
qc.x(output_qubit); qc.h(output_qubit)   # |0⟩ -X-> |1⟩ -H-> |−⟩ (the original qc.initialize([1, -1]/np.sqrt(2), ...) starts with a reset, which not every backend supports)

# Initialize qubits in state |s>
qc.h(var_qubits)                         # H on every qubit of the register: uniform superposition of the 16 assignments
qc.barrier()  # for visual separation

## First Iteration
# Apply our oracle
sudoku_oracle(qc, clause_list, clause_qubits)
qc.barrier()  # for visual separation
# Apply our diffuser
qc.append(diffuser(4), [0,1,2,3])        # the general diffuser of Section 3, on the 4 variable qubits only

## Second Iteration
sudoku_oracle(qc, clause_list, clause_qubits)
qc.barrier()  # for visual separation
# Apply our diffuser
qc.append(diffuser(4), [0,1,2,3])

# Measure the variable qubits
qc.measure(var_qubits, cbits)            # v_i -> cbits_i; the clause and output qubits are not measured

display(draw(qc)); print(f"↑ the full sudoku search: |−⟩ on 'out', |s⟩ on v, then (oracle | diffuser) twice and the measurement of v — {qc.num_qubits} qubits, {dict(qc.count_ops())}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.x(output_qubit); qc.h(output_qubit)` | prepares \|−⟩ = H\|1⟩ for the phase kickback; the original `initialize([1, -1]/np.sqrt(2))` gives the same state but begins with a `reset`, an instruction IQM hardware does not support |
| `qc.h(var_qubits)` | a gate applied to a whole register acts on each of its qubits |
| two iterations | N = 16, M = 2: θ = 20.7°, so k = 2 is optimal (box after the run) |
| `diffuser(4)` on `[0,1,2,3]` | the diffuser acts on the variables only: the clause qubits are back to \|0000⟩ and `out` stays in \|−⟩, so they factor out |
| `qc.measure(var_qubits, cbits)` | only the 4 sudoku cells are read out |

</details>


In [ ]:
# Run on the IQM backend chosen by TARGET, with the ideal Aer result as a reference
counts_sudoku, transpiled_qc = run_counts(qc, TARGET, shots=1024, return_transpiled=True)
print(f"transpiled for IQM: {qc.num_qubits} of {transpiled_qc.num_qubits} qubits used, {transpiled_qc.size()} native instructions (cap 10 000), depth {transpiled_qc.depth()}, {dict(transpiled_qc.count_ops())}")
qasm_simulator = AerSimulator()                                   # ideal (noise-free) reference
ideal_sudoku = qasm_simulator.run(generate_preset_pass_manager(1, qasm_simulator).run(qc), shots=1024).result().get_counts()
print("most frequent on TARGET:", sorted(counts_sudoku.items(), key=lambda kv: -kv[1])[:4])
print("most frequent, ideal   :", sorted(ideal_sudoku.items(), key=lambda kv: -kv[1])[:4])
plot_histogram([ideal_sudoku, counts_sudoku], legend=["ideal (Aer)", TARGET], figsize=(12, 4))   # two dicts -> grouped bars


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `run_counts(qc, TARGET, shots=1024, return_transpiled=True)` | the sudoku search on `TARGET`: counts and the transpiled circuit |
| `transpiled_qc.size()`, `count_ops()` | 9 of the 20 qubits, roughly 450–550 native instructions (the number changes slightly between runs: the routing is stochastic), about 170–200 of them `cz` — well below the 10 000 cap, so the circuit runs on `TARGET` |
| `ideal_sudoku` | the same circuit on the noise-free Aer simulator: the reference that the text below describes |
| `plot_histogram([d1, d2], legend=[...])` | grouped bars: the ideal run puts ≈ 94.5 % on `0110` and `1001`; the `fake_apollo` run is almost flat (each outcome ≈ 1/16 of the shots) — the noise of ~200 `cz` gates has erased the interference (note below) |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**How many iterations for $M$ solutions out of $N$.** From the geometric picture (box in Section 2), $\sin\theta = \sqrt{M/N}$ and $P(k) = \sin^2((2k+1)\theta)$; the best integer $k$ brings $(2k+1)\theta$ closest to $90^\circ$:

$$k_{\rm opt} = \operatorname{round}\Big(\frac{\pi}{4\theta} - \frac12\Big) \;\approx\; \frac{\pi}{4}\sqrt{\frac NM}\quad(N \gg M),$$

the $t = \lfloor\frac\pi4\sqrt{N/m}\rfloor$ of the introduction.

**The sudoku.** $N = 2^4 = 16$, $M = 2$: $\sin\theta = \sqrt{2/16} = 0.354$, $\theta = 20.7^\circ$, $\pi/(4\theta) = 2.17$, so $k_{\rm opt} = \operatorname{round}(1.67) = 2$ — the two iterations of the circuit — and

$$P = \sin^2(5\theta) = \sin^2 103.5^\circ = 0.945 ,$$

i.e. about $968$ of $1024$ shots on `0110` and `1001` together (≈ 484 each), the other 14 outcomes sharing the remaining 5.5 % (≈ 4 shots each): what the ideal simulation prints. One iteration would give $\sin^2 62.1^\circ = 0.78$, three would overshoot to $\sin^2 144.9^\circ = 0.33$ (task 2 of the In-depth analysis).

**All the searches of this notebook.**

| search | $N$ | $M$ | $\theta$ | $k_{\rm opt}$ | $P(k_{\rm opt})$ |
|---|---|---|---|---|---|
| 2 qubits, $w = 11$ | 4 | 1 | 30° | 1 | 1 (exact) |
| 3 qubits, $\{101, 110\}$ | 8 | 2 | 30° | 1 | 1 (exact) |
| sudoku | 16 | 2 | 20.7° | 2 | 0.945 |
| triangle, W-state search space | 4 | 1 | 30° | 1 | 1 (exact) |

Whenever $M/N = 1/4$ the angle is exactly $30^\circ$ and one iteration lands on the solutions. In the triangle problem (Section 4.2) the W-state preparation shrinks the search space from the $2^4 = 16$ bitstrings ($\theta = 14.5^\circ$, 3 iterations) to the $\binom43 = 4$ three-node subgraphs — that is what turns 3 iterations into 1.

</details>


> **Note — the sudoku on IQM hardware.** The circuit fits: it needs 9 qubits (the 20-qubit Garnet / `fake_apollo` has room to spare) and, once transpiled, roughly 450–550 native instructions — far below the 10 000-instruction cap of IQM Resonance. It therefore runs on `TARGET` like every other circuit of this notebook, with no fallback to Aer. But fitting is not working: the 4-controlled X gates of the oracle and the 3-controlled X gates of the diffusers become about 170–200 `cz` gates once decomposed and routed on the square lattice, and with that many two-qubit gates the noise of `fake_apollo` washes out the interference — the two solutions no longer stand out of a nearly uniform histogram. The ideal Aer histogram is plotted next to it as the reference that the text below describes; the In-depth analysis (task 3) puts a number on the cost. The triangle search of Section 4.2 is in the same situation.


There are two bit strings with a much higher probability of measurement than any of the others, `0110` and `1001`. These correspond to the assignments:
```
v0 = 0
v1 = 1
v2 = 1
v3 = 0
```
and
```
v0 = 1
v1 = 0
v2 = 0
v3 = 1
```
which are the two solutions to our sudoku! The aim of this section is to show how we can create Grover oracles from real problems. While this specific problem is trivial, the process can be applied (allowing large enough circuits) to any decision problem. To recap, the steps are:



&nbsp;

<!-- ::: q-block.exercise -->

### Your turn

1. Create a reversible classical circuit that identifies a correct solution
2. Use phase kickback and uncomputation to turn this circuit into an oracle
3. Use Grover's algorithm to solve this oracle

<!-- ::: -->


## 4.2 The Triangle-finding Problem Using Grover <a id='tri'></a>


One of the famous [graph theory](gloss:gt) problems is the [triangle-finding problem](https://en.wikipedia.org/wiki/Triangle-free_graph). In the triangle-finding problem, we are given a graph that may or may not contain a triangle. Our task is to find the triangle/s within the graph and point out the nodes that form the triangle. For example, the graph below is a 4-node graph with a triangle between nodes `0`,`1`, and `2`.     


![image12](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_tri.png)


We can apply Grover's algorithm to this problem, we are going to give the algorithm a list of _edges_ and the number of nodes in the graph. The algorithm, then, will do the rest. It will try to check if there's a triangle in the graph or not, and if so it will mark the nodes forming that triangle.   


Now, let's go through Grover's algorithm steps and see how can we construct each step to solve the triangle finding problem. But first, let's define our input, which is the list of edges.

In [ ]:
#Edges list
edges =[(0, 1), (0, 2), (1, 2), (2, 3)]   # a list of tuples (node, node): the graph of the figure, triangle 0-1-2 plus the edge 2-3
#Number of nodes
n_nodes = 4


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `edges = [(0, 1), ...]` | the graph as a list of tuples (pairs of node indices); tuples are immutable lists |
| `n_nodes = 4` | one qubit per node: bit i = 1 means node i is in the candidate subgraph |

</details>


### 4.2.1 The state preparation

To solve the problem, let's first focus on the example above, which is the case of finding a triangle in a 4-node graph. to do that, we need to go over all subgraphs within our graph and check if any of them a triangle. To do that we will need 4-qubits, each qubit represents a node in the graph. The state of the qubit will indicate whether the node in any subgraph or not. For example, in the graph above, the triangle is between nodes 0,1, and 2, we can rephrase that using the state `1110`. The nodes with state 1 are in the subgraph (triangle) and the node with state 0 is not.   

For the state preparation, we will need to create a superposition of all possible states `0000`, to `1111`, which can be done simply using `4` Hadamard gates. Doing so, will we need to rotate over the oracle and diffusion `3` times.  
But, if each `1` in the state represent an active node, we don't really need to look through the entire Hilbert space, we only need to look through the subgraphs with three nodes.    

This is a good example of using another type of symmetric states to prepare the search space. Here since we only need to consider states with three 1's, we can think of another way to form our search space. One way to create a superposition over only states with three active nodes is through using the W-state followed by `4 NOT gates`. This will decrease the number of rotations needed from `3` to `1`.

So, we first need to implement the W-state. W-states have the form:

$$

|W\rangle = {\frac{1}{{\sqrt{n}}}}(|100...0\rangle + ... + |01...0\rangle + |00...01\rangle)

$$

In our case, we need to construct $|W_{3}\rangle$ states as described in reference 6.

In [ ]:
#We used the W state implementation from W state in reference 6
def control_rotation (qcir,cQbit,tQbit,theta):
    """ Create an intermediate controlled rotation using only unitary gate and controlled-NOT

    Args:
    qcir: QuantumCircuit instance to apply the controlled rotation to.
    cQbit: control qubit.
    tQbit: target qubit.
    theta: rotation angle.

    Returns:
    A modified version of the QuantumCircuit instance with control rotation applied.

    """
    theta_dash = math.asin(math.cos(math.radians(theta/2)))   # math.radians: degrees -> radians
    qcir.u(theta_dash,0,0,tQbit)                               # u(θ, φ, λ) with φ = λ = 0 is a real rotation RY(θ)
    qcir.cx(cQbit,tQbit)
    qcir.u(-theta_dash,0,0,tQbit)                              # RY(θ') · CX · RY(−θ'): a rotation that depends on the control
    return qcir


def wn (qcir,qbits):
    """ Create the W-state using the control-rotation function.

    Args:
    qcir: QuantumCircuit instance used to construct the W-state.
    qbits: the qubits used to construct the W-state.

    Returns:
    A modified version of the QuantumCircuit instance with the W-state construction gates.

    """
    for i in range(len(qbits)):
        if i == 0:
            qcir.x(qbits[0])                                   # start from |10…0⟩: the single excitation sits on the first qubit
            qcir.barrier()
        else:
            p = 1/(len(qbits)-(i-1))                           # probability to leave the excitation where it is: 1/n, 1/(n−1), ...
            theta = math.degrees(math.acos(math.sqrt(p)))
            theta = 2* theta
            qcir = control_rotation(qcir,qbits[i-1],qbits[i],theta)
            qcir.cx(qbits[i],qbits[i-1])                       # move the excitation one qubit further along, with the right amplitude
            qcir.barrier()
    return qcir,qbits                                          # a tuple: the circuit and the qubits

sub_qbits = QuantumRegister(n_nodes)
sub_cir = QuantumCircuit(sub_qbits, name="state_prep")
sub_cir, sub_qbits = wn(sub_cir, sub_qbits)                   # tuple unpacking of the two return values
sub_cir.x(sub_qbits)                                          # X on all: one 1 among four 0s -> one 0 among four 1s (three nodes chosen)
stat_prep = sub_cir.to_instruction()                          # the preparation as one instruction (a box)
inv_stat_prep = sub_cir.inverse().to_instruction()            # its inverse, needed by the diffuser

display(draw(sub_cir)); print("↑ state preparation: W-state on 4 qubits (X, then controlled rotations + CNOTs that share out the single 1), then X on every qubit")
probs = Statevector(sub_cir).probabilities_dict()             # exact probabilities of the prepared state
print("prepared state:", {str(k): round(float(v), 3) for k, v in probs.items() if v > 1e-9})   # dict comprehension: drop the ~0 entries


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `control_rotation` | RY(θ′) · CX · RY(−θ′) on the target: a rotation conditioned on the control, built from single-qubit gates and one CNOT |
| `wn(qcir, qbits)` | the W state (\|1000⟩ + \|0100⟩ + \|0010⟩ + \|0001⟩)/2: start from one excitation and share it out with amplitudes √(1/n), √(1/(n−1)), … |
| `sub_cir.x(sub_qbits)` | complementing every bit turns 'exactly one 1' into 'exactly three 1s': the uniform superposition of the 4 three-node subgraphs |
| `to_instruction()`, `inverse()` | the preparation and its inverse as boxes: the diffuser for a non-uniform start is A · (reflection about \|0⟩) · A† |
| printed state | `0111`, `1011`, `1101`, `1110` with probability 0.25 each: N = 4 candidates instead of 16 |

</details>


### 4.2.2 The oracle

The oracle is what's gonna mark the correct answer. In these cases, the oracle needs to take every subgraph and count the number of edges in that subgraph. If the number of edges is `3`, then we have a triangle, if not, it will proceed to the next subgraph.    

![image13](https://raw.githubusercontent.com/Qiskit/textbook/main/notebooks/ch-algorithms/images/grover_tri_oracle.png)

For every edge in the graph, we will need one or two `CNOT` gates. These `CNOT` gates will apply to two ancillary qubits, that should be in state `11` if a triangle is found. The number of ancillary qubits here is two because a triangle has `3 edges` that is $11_{b}$. Then the final step in the oracle is to apply one more `Toffoli`, that will only be active if a triangle is found by changing the state of another qubit, let's call it, `tri_flag`, to `1`.

&nbsp;

<!-- ::: q-block.reminder -->

### Reminder

<details>
    <summary>Oracle</summary>
    The oracle's job is to verify and mark the correct answer. So, when you construct an oracle, you're basically building a circuit to verify certain conditions.
</details>

<!-- ::: -->

In [ ]:
def edge_counter(qc,qubits,anc,flag_qubit,k):
    bin_k = bin(k)[2:][::-1]                     # binary digits of k, least significant first ('0b11'[2:] = '11', [::-1] reverses)
    l = []
    for i in range(len(bin_k)):
        if int(bin_k[i]) == 1:
            l.append(qubits[i])                  # control on the counter bits that must be 1
    qc.mcx(l,flag_qubit)                         # flip the flag if the counter equals k (the original mct(l, flag, [anc]) was removed in Qiskit 1.0; mcx needs no ancilla)


def oracle(n_nodes, edges, qc, nodes_qubits, edge_anc, ancilla, neg_base):
    k = 3 #k is the number of edges, in case of a triangle, it's 3
    #1- edge counter
    #forward circuit
    qc.barrier()
    qc.ccx(nodes_qubits[edges[0][0]],nodes_qubits[edges[0][1]],edge_anc[0])   # first edge present -> counter = 1
    for i in range(1,len(edges)):
        qc.mcx([nodes_qubits[edges[i][0]],nodes_qubits[edges[i][1]],edge_anc[0]], edge_anc[1])   # carry: both nodes in and low bit 1 -> flip the high bit
        qc.ccx(nodes_qubits[edges[i][0]],nodes_qubits[edges[i][1]],edge_anc[0])                  # then add 1 to the low bit
    #----------------------------------------------------------------------------------------------------------
     #Edges check Qubit
    edg_k = int((k/2)*(k-1))                     # k(k−1)/2 = 3 edges among k = 3 nodes
    edge_counter(qc,edge_anc,ancilla[0],neg_base[0],edg_k)   # flag (in |−⟩) flipped iff the 2-bit counter reads 3 = 11b -> phase −1
    #----------------------------------------------------------------------------------------------------------

    #4- Reverse edge count
    for i in range(len(edges)-1,0,-1):           # uncompute the counter, in reverse order
        qc.ccx(nodes_qubits[edges[i][0]],nodes_qubits[edges[i][1]],edge_anc[0])
        qc.mcx([nodes_qubits[edges[i][0]],nodes_qubits[edges[i][1]],edge_anc[0]], edge_anc[1])
    qc.ccx(nodes_qubits[edges[0][0]],nodes_qubits[edges[0][1]],edge_anc[0])
    qc.barrier()


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `bin(k)[2:][::-1]` | the binary digits of k as a string, least significant first: `bin(3)` = `'0b11'` |
| `ccx` + `mcx` per edge | a 2-bit counter (`edge_anc`) incremented by 1 for every edge whose two nodes are both in the subgraph: ccx flips the low bit, the 3-controlled X carries into the high bit |
| `edge_counter(..., edg_k)` | flips the \|−⟩ flag (`neg_base`) iff the counter equals 3: phase kickback gives −1 to the triangles |
| reverse loop | uncomputation: the counter is reset to \|00⟩ so that it does not stay entangled with the nodes (same reason as in the sudoku) |
| `qc.mct(..., [anc])` → `qc.mcx(...)` | `mct` was removed in Qiskit 1.0; `mcx` without ancillas is synthesised automatically (the ancilla register is still used by the diffuser) |

</details>


### 4.2.3 The diffusion operator

As we said before, the diffusion operator construction depends on the type of state preparation we used, in this case, the W-state. So, we need the inverses W-state, a multi-controlled Z gate and the original W-state to form the diffusion operator.

In [ ]:
def cnz(qc, num_control, node, anc):
    """Construct a multi-controlled Z gate

    Args:
    num_control :  number of control qubits of cnz gate
    node :             node qubits
    anc :               ancillaly qubits
    """
    if num_control>2:
        qc.ccx(node[0], node[1], anc[0])                       # anc0 = node0 AND node1
        for i in range(num_control-2):
            qc.ccx(node[i+2], anc[i], anc[i+1])                # anc_{i+1} = anc_i AND node_{i+2}: a chain of ANDs
        qc.cz(anc[num_control-2], node[num_control])           # the phase −1 if all controls and the target are 1
        for i in range(num_control-2)[::-1]:
            qc.ccx(node[i+2], anc[i], anc[i+1])                # undo the chain (uncompute the ancillas)
        qc.ccx(node[0], node[1], anc[0])
    if num_control==2:
        qc.h(node[2])
        qc.ccx(node[0], node[1], node[2])                      # H·CCX·H = CCZ
        qc.h(node[2])
    if num_control==1:
        qc.cz(node[0], node[1])


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `cnz(qc, num_control, node, anc)` | a multi-controlled Z built by hand from Toffolis: compute the AND of the controls into the ancillas, apply a CZ, uncompute |
| `range(num_control-2)[::-1]` | the same loop in reverse order: the uncomputation mirrors the computation |
| three cases | 3 or more controls use the ancilla chain; 2 controls are H·CCX·H; 1 control is a plain CZ |

</details>


In [ ]:
def grover_diff(qc, nodes_qubits,edge_anc,ancilla,stat_prep,inv_stat_prep):
    qc.append(inv_stat_prep,qargs=nodes_qubits)                # A†: the W-based superposition -> |0000⟩
    qc.x(nodes_qubits)                                         # |0000⟩ -> |1111⟩
    #====================================================
        #3 control qubits Z gate
    cnz(qc,len(nodes_qubits)-1,nodes_qubits[::-1],ancilla)    # −1 on |1111⟩ (3 controls + target)
    #====================================================
    qc.x(nodes_qubits)
    qc.append(stat_prep,qargs=nodes_qubits)                    # A: back to the search space


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `inv_stat_prep` … `stat_prep` | the diffuser for a general preparation A: A · (reflection about \|0000⟩) · A† = reflection about A\|0000⟩, the W-based start state |
| `nodes_qubits[::-1]` | the node register reversed (any order works for a symmetric multi-controlled Z) |

</details>


### 4.2.4 Putting it all together

Now that we have all the components of the algorithm built and running, we can put them together.

In [ ]:
# Grover algo function
def grover(n_nodes,stat_prep,inv_stat_prep):
    #N = 2**n_nodes  # for optimal iterations count if the state prep is done using only H gates.
    N = math.comb(n_nodes, 3) #Since we are using W-state to perform initial preparation.   math.comb(4, 3) = 4
    nodes_qubits = QuantumRegister(n_nodes, name='nodes')
    edge_anc = QuantumRegister(2, name='edge_anc')               # the 2-bit edge counter
    ancilla = QuantumRegister(n_nodes-2, name = 'cccx_diff_anc')  # ancillas of the hand-made multi-controlled Z
    neg_base = QuantumRegister(1, name='check_qubits')           # the phase-kickback flag
    class_bits = ClassicalRegister(n_nodes, name='class_reg')
    tri_flag = ClassicalRegister(3, name='tri_flag')             # declared by the author but never measured (stays 000)
    qc = QuantumCircuit(nodes_qubits, edge_anc, ancilla, neg_base, class_bits, tri_flag)
    # Initialize qunatum flag qubits in |-> state
    qc.x(neg_base[0])
    qc.h(neg_base[0])
    # Initializing i/p qubits in superposition
    qc.append(stat_prep,qargs=nodes_qubits)
    qc.barrier()
    # Calculate iteration count
    iterations = math.floor(math.pi/4*math.sqrt(N))             # floor(π/4 · 2) = 1
    # Calculate iteration count
    for i in np.arange(iterations):
        qc.barrier()
        oracle(n_nodes, edges, qc, nodes_qubits, edge_anc,  ancilla, neg_base)
        qc.barrier()
        grover_diff(qc, nodes_qubits,edge_anc,ancilla,stat_prep,inv_stat_prep)
    qc.measure(nodes_qubits,class_bits)
    return qc


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `math.comb(n_nodes, 3)` | the number of 3-node subsets, C(4, 3) = 4: the size of the W-based search space (with H on every qubit it would be 2⁴ = 16) |
| five registers | `nodes` (4), `edge_anc` (2), `cccx_diff_anc` (2), `check_qubits` (1): 9 qubits; classical `class_reg` (4) and `tri_flag` (3) |
| `math.floor(math.pi/4*math.sqrt(N))` | the iteration formula of the introduction: ⌊π/4 · √4⌋ = ⌊1.57⌋ = 1 |
| `tri_flag` | a classical register that nothing writes: it shows up as `000` in the Aer counts keys |

</details>


Now, let's run the code and plot the histogram to see if our algorithm works as expected.

In [ ]:
qc = grover(n_nodes,stat_prep,inv_stat_prep)
print(f"{qc.num_qubits} qubits, {qc.size()} instructions, {dict(qc.count_ops())}")
display(draw(qc, fold=40)); print("↑ the triangle search: |−⟩ flag, W-based preparation, then one iteration = edge-counting oracle (computed and uncomputed) + W-state diffuser, and the measurement of the 4 node qubits")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `grover(n_nodes, stat_prep, inv_stat_prep)` | builds the whole circuit with the iteration count computed inside |
| `count_ops()` | the gate inventory: Toffolis and 3-controlled X of the counter, the boxes of the preparation and its inverse |
| `draw(qc, fold=40)` | a long circuit: `fold=40` would wrap it after 40 columns (this one fits in a single row) |

</details>


In [ ]:
# Run on the IQM backend chosen by TARGET, with the ideal Aer result as a reference
res, transpiled_qc = run_counts(qc, TARGET, shots=5000, return_transpiled=True)
print(f"transpiled for IQM: {transpiled_qc.size()} native instructions (cap 10 000), depth {transpiled_qc.depth()}, {transpiled_qc.count_ops().get('cz', 0)} cz")
qasm_simulator = AerSimulator()
ideal = qasm_simulator.run(generate_preset_pass_manager(1, qasm_simulator).run(qc), shots=5000).result().get_counts()
ideal = {k.split()[-1]: v for k, v in ideal.items()}             # Aer writes 'tri_flag class_reg'; keep the class_reg part, as in the IQM keys
print("ideal (Aer):", ideal)
print("most frequent on TARGET:", sorted(res.items(), key=lambda kv: -kv[1])[:4])
plot_histogram([ideal, res], legend=["ideal (Aer)", TARGET], figsize=(12, 4))


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `run_counts(qc, TARGET, shots=5000, return_transpiled=True)` | the triangle search on `TARGET`; `res` has 4-bit keys (the measured `class_reg`), the never-written `tri_flag` register does not appear |
| `k.split()[-1]` | the ideal Aer keys `'000 0111'` reduced to their last group, so that both dicts use the same keys |
| `transpiled_qc.size()` | about 800 native instructions, ~300 of them `cz`: below the 10 000 cap, but deep |
| the two histograms | ideal: all 5000 shots on `0111`, the triangle; on `fake_apollo` the shots are spread almost uniformly over the 16 strings and `0111` no longer stands out: too many noisy two-qubit gates |

</details>


<!-- ::: q-block.exercise -->

### Your turn

Can you extend this problem to find a triangle in any size graph?

Try in [IBM Quantum Lab](https://quantum-computing.ibm.com/lab)

<!-- ::: -->


## 🔬 In-depth analysis

1. **The author's Problems 1 and 3: a Grover solver and the iteration count.** Write `grover_solver(oracle, iterations)` (Problem 3) with `initialize_s` and `diffuser`, and apply it to `grover_problem_oracle(4, variant=2)` (one hidden solution, $N = 16$). With `Statevector`, compute the largest outcome probability for $k = 0, \dots, 6$ iterations and compare it with $\sin^2((2k+1)\theta)$, $\sin\theta = 1/4$. What is the smallest $k$ above 90 % (Problem 1a), which $k$ is optimal, and what happens beyond it (Problem 1c)? Sample the optimal circuit with `run_circuit` to read off the solution.
2. **Counting the sudoku solutions, then the iterations.** Count classically the assignments that satisfy all four clauses of `clause_list` ($M$), deduce $\theta$, $k_{\rm opt}$ and the predicted success probability for $N = 16$, and check them on the state vector of the sudoku circuit with $k = 1, 2, 3$ iterations (marginal probability of the variable register: `probabilities_dict(qargs=[0, 1, 2, 3])`).
3. **What the sudoku costs on hardware.** Rewrite the 2-iteration sudoku circuit with `to_native` (the native gates of this edition's hardware, defined in Section 2.1.1), count the native instructions and the two-qubit gates, compare with the 10 000-instruction cap of IQM Resonance, and measure the success fraction (shots on `0110` or `1001`) with `run_circuit`. What does the two-qubit gate count predict for a real device?

Write your code in the empty cell below; a reference solution follows it. Try first, then compare.


In [ ]:
# 🔬 In-depth analysis — your code here



In [ ]:
# Reference solution
# (1) the author's Problems 1 and 3: grover_solver, and the number of iterations for 1 solution out of N = 16
def grover_solver(oracle, iterations):
    """Problem 3: the Grover circuit for a phase-oracle gate, with the given number of iterations (no measurement)."""
    n_q = oracle.num_qubits
    circ = QuantumCircuit(n_q)
    initialize_s(circ, range(n_q))                                     # |s⟩
    for _ in range(iterations):                                        # _ = a loop variable we do not use
        circ.append(oracle, range(n_q))                                # U_ω
        circ.append(diffuser(n_q), range(n_q))                         # U_s
    return circ

oracle_p1 = grover_problem_oracle(4, variant=2)                         # one hidden solution
theta1 = np.arcsin(np.sqrt(1 / 16))                                     # sin θ = √(M/N)
probs1 = [Statevector(grover_solver(oracle_p1, k)).probabilities() for k in range(7)]   # exact probabilities of the 16 outcomes (index = bitstring as an integer)
sol1 = int(np.argmax(probs1[3]))                                        # the hidden solution: the outcome that dominates near the optimum
for k in range(7):
    print(f"(1) k = {k}: P({sol1:04b}) = {probs1[k][sol1]:.3f}; sin²((2k+1)θ) = {np.sin((2 * k + 1) * theta1) ** 2:.3f}")
k90 = next(k for k in range(7) if np.sin((2 * k + 1) * theta1) ** 2 > 0.9)   # next(generator) = the first k that qualifies
k_best = round(np.pi / (4 * theta1) - 0.5)
circ_best = grover_solver(oracle_p1, k_best); circ_best.measure_all()
top = max(run_circuit(circ_best, shots=1024).items(), key=lambda kv: kv[1])  # the most frequent (bitstring, count)
print(f"(1) above 90 % from k = {k90}; optimum k = {k_best}; beyond it the state rotates past the solution and P falls (to 2 % at k = 6)")
print(f"(1) sampled at k = {k_best}: most frequent outcome {top[0]} ({top[1]} of 1024 shots)")

# (2) the sudoku: M by brute force, then θ, k_opt and the check on the state vector
def satisfies(x):
    v = [(x >> q) & 1 for q in range(4)]                               # v[q] = bit q of x
    return all(v[a] != v[b] for a, b in clause_list)                    # every clause: the two cells differ
solutions = [format(x, "04b") for x in range(16) if satisfies(x)]
M, N = len(solutions), 16
theta2 = np.arcsin(np.sqrt(M / N)); k_opt = round(np.pi / (4 * theta2) - 0.5)
print(f"(2) solutions {solutions}: M = {M}, θ = {np.degrees(theta2):.2f}°, k_opt = {k_opt}, predicted P = {np.sin((2 * k_opt + 1) * theta2) ** 2:.3f}")

def sudoku_grover(k, measure=True):
    """The circuit of Section 4.1.3 with k iterations."""
    circ = QuantumCircuit(var_qubits, clause_qubits, output_qubit, cbits)
    circ.x(output_qubit); circ.h(output_qubit)                         # out = |−⟩
    circ.h(var_qubits)                                                 # |s⟩ on the 4 variables
    for _ in range(k):
        sudoku_oracle(circ, clause_list, clause_qubits)
        circ.append(diffuser(4), [0, 1, 2, 3])
    if measure: circ.measure(var_qubits, cbits)
    return circ

for k in (1, 2, 3):
    pv = Statevector(sudoku_grover(k, measure=False)).probabilities_dict(qargs=[0, 1, 2, 3])   # marginal on v3 v2 v1 v0
    print(f"(2) k = {k}: P(solution) = {sum(pv.get(s, 0) for s in solutions):.3f} (theory {np.sin((2 * k + 1) * theta2) ** 2:.3f})")

# (3) the hardware cost of the 2-iteration sudoku circuit
sudoku2 = sudoku_grover(2)
native = to_native(sudoku2)
ops = dict(native.count_ops()); two_q = sum(v for g, v in ops.items() if g in ("cx", "cz", "ecr"))   # the two-qubit gates of either hardware family
print(f"(3) native: {native.size()} instructions (IQM Resonance cap: 10 000), depth {native.depth()}, {two_q} two-qubit gates; {ops}")
counts3 = run_circuit(sudoku2, shots=1024)
succ = sum(counts3.get(s, 0) for s in solutions) / 1024
print(f"(3) success fraction on this edition's backend: {succ:.3f} (ideal {np.sin(5 * theta2) ** 2:.3f}, random guessing {M / N:.3f})")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `grover_solver(oracle, iterations)` | the author's Problem 3: \|s⟩, then `iterations` times oracle + diffuser, for any phase-oracle gate (the number of qubits is read from `oracle.num_qubits`) |
| `Statevector(...).probabilities()`, `np.argmax(...)` | the exact probabilities as an array indexed by the bitstring's integer value; the hidden solution is the index that dominates at k = 3, and `f"{i:04b}"` writes it as 4 bits |
| `next(k for k in ... if ...)` | a generator expression and `next`: the first k satisfying the condition (Problem 1a) |
| `satisfies(x)`, `all(...)` | the classical verifier: true if every clause pair has different values — counting its solutions gives M = 2 |
| `probabilities_dict(qargs=[0, 1, 2, 3])` | the marginal distribution of the 4 variable qubits, summing over clause and output qubits |
| `to_native(sudoku2)`, two-qubit count | the circuit in the hardware's native gates (cx on IBM, cz on IQM); each two-qubit gate costs ~1 % error on today's devices, so a few hundred of them wipe out the interference |
| `run_circuit(sudoku2)` | the measured success fraction on this edition's backend: ideal Aer in the IBM edition, `TARGET` (noisy `fake_apollo` by default) in the IQM edition |

</details>


## 5. References <a id='references'></a>

1. L. K. Grover (1996), "A fast quantum mechanical algorithm for database search", Proceedings of the 28th Annual ACM Symposium on the Theory of Computing (STOC 1996), [doi:10.1145/237814.237866](http://doi.acm.org/10.1145/237814.237866), [arXiv:quant-ph/9605043](https://arxiv.org/abs/quant-ph/9605043)
2. C. Figgatt, D. Maslov, K. A. Landsman, N. M. Linke, S. Debnath & C. Monroe (2017), "Complete 3-Qubit Grover search on a programmable quantum computer", Nature Communications, Vol 8, Art 1918, [doi:10.1038/s41467-017-01904-7](https://doi.org/10.1038/s41467-017-01904-7), [arXiv:1703.10535 ](https://arxiv.org/abs/1703.10535)
3. I. Chuang & M. Nielsen, "Quantum Computation and Quantum Information", Cambridge: Cambridge University Press, 2000.   
4. Marconi, C., Aloy, A., Tura, J., & Sanpera, A. (2021). Entangled symmetric states and copositive matrices. Quantum, 5, 561.
5. R. Prevedel, G. Cronenberg, M. S. Tame, M. Paternostro, P. Walther, M. S. Kim, and A. Zeilinger, “Experimental realization of dicke states of up to six qubits for multiparty quantum networking,” Physical Review Letters, vol. 103, no. 2, Jul 2009.   
6. D. Cruz, R. Fournier, F. Gremion, A. Jeannerot, K. Komagata, T. Tosic, J. Thiesbrummel, C. L. Chan, N. Macris, M.-A. Dupertuis et al., “Efficient quantum algorithms for ghz and w states, and implementation on the ibm quantum computer,” Advanced Quantum Technologies, vol. 2, no. 5-6, p.1900015, 2019. https://doi.org/10.1002/qute.201900015
7. Magniez, F., Santha, M., & Szegedy, M. (2007). Quantum algorithms for the triangle problem. SIAM Journal on Computing, 37(2), 413-424.

In [ ]:
# The original used qiskit.tools.jupyter and %qiskit_version_table, removed in Qiskit 1.0: print the versions directly
import sys, qiskit, qiskit_aer
print(f"Python {sys.version.split()[0]} | qiskit {qiskit.__version__} | qiskit-aer {qiskit_aer.__version__}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qiskit.__version__` | the installed version of each package; `sys.version.split()[0]` is the Python version number |
| `%qiskit_version_table` removed | the Jupyter magic of `qiskit.tools.jupyter` no longer exists in Qiskit ≥ 1.0 |

</details>
